## 📦 패키지 설치
> Cursor / VS Code 에서 처음 실행할 때 아래 셀을 먼저 실행하세요.

In [1]:
# 필요 패키지 설치 (이미 설치된 경우 skip)
%pip install pandas --quiet
%pip install ipython --quiet
%pip install jupyter --quiet
%pip install ipywidgets --quiet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\cho01\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\cho01\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\cho01\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\cho01\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


---

# 🎵 청음 문제 생성 알고리즘 (단일음 + 음정)

## 전체 구조

```
① category / ② course / ③ part / ④ step (DB)
⑤ answer_type 규칙 / ⑥ difficulty_level 규칙
─────────────────────────────────────────────
음 유틸 (MIDI ↔ 음이름) + 음정 유틸 (기호 ↔ 반음수)
SingleNoteGenerator  ←  단일음 문제 생성
IntervalGenerator    ←  음정 문제 생성
simulate()           ←  통합 시뮬레이터 (위젯 UI)
run_exhaustive_validation()  ←  전수 검증기
```


---
## 1. Config & 공통 유틸

In [2]:
import random
import pandas as pd
from typing import Optional
from IPython.display import display

try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

# ── 시스템 제약 상수 ────────────────────────────────────────────────
MIDI_MIN     = 48      # C3
MIDI_MAX     = 84      # C6
MAX_RETRY    = 3
DEFAULT_SEED = None

# ── pandas 출력 스타일 헬퍼 ─────────────────────────────────────────
def show(df: pd.DataFrame, title: str = '', color: str = '#2C3E50') -> None:
    if title:
        print(f'\n── {title} ──')
    styled = (
        df.style
        .set_table_styles([
            {'selector': 'th', 'props': [
                ('background-color', color), ('color', 'white'),
                ('font-weight', 'bold'), ('text-align', 'center'),
                ('padding', '6px 10px'),
            ]},
            {'selector': 'td', 'props': [('padding', '4px 10px')]},
        ])
        .set_properties(**{'text-align': 'left'})
    )
    display(styled)


---
## 2. 테이블 선언 (DB 묘사)

> 실제 서비스에서는 이 데이터가 DB에 저장되고 쿼리로 조회됩니다.  
> 여기서는 **"DB에 이런 형태로 담겨있다"** 는 것을 전달하기 위해 DataFrame으로 선언합니다.
>
> **계층 참조 관계**
> ```
> ① category ← ② course ← ③ part ← ④ 커리큘럼(step)
>                                        ↓ answer_type     → ⑤ 규칙 참조
>                                        ↓ difficulty_level → ⑥ 규칙 참조 (동적)
> ```

In [3]:
# ══════════════════════════════════════════════════════════════════════
# ① category 테이블
# ══════════════════════════════════════════════════════════════════════
CATEGORY_DATA = [
    ('CAT_SN',  '단일음'),
    ('CAT_INT', '음정'),   # INTerval — 코드에서 문자열로만 사용
]
df_category = pd.DataFrame(CATEGORY_DATA, columns=['category_id', 'category_name'])
show(df_category, '① category 테이블', color='#1E8449')



── ① category 테이블 ──


,category_id,category_name
0,CAT_SN,단일음
1,CAT_INT,음정


In [4]:
# ══════════════════════════════════════════════════════════════════════
# ② course 테이블
# ══════════════════════════════════════════════════════════════════════
COURSE_DATA = [
    # category_id,  course_id,           course_name
    ('CAT_SN',      'CAT_SN_SC01',       '7음계'),
    ('CAT_SN',      'CAT_SN_SC02',       '12음계'),
    ('CAT_INT',     'CAT_INT_SC01',      '코스1: 1도, 2도'),
    ('CAT_INT',     'CAT_INT_SC02',      '코스2: 2도, 3도'),
    ('CAT_INT',     'CAT_INT_SC03',      '코스3: 3도, 4도'),
    ('CAT_INT',     'CAT_INT_SC04',      '코스4: 4도, 5도'),
    ('CAT_INT',     'CAT_INT_SC05',      '코스5: 3도, 6도'),
    ('CAT_INT',     'CAT_INT_SC06',      '코스6: 2도, 7도'),
]
df_course = pd.DataFrame(COURSE_DATA, columns=['category_id', 'course_id', 'course_name'])
show(df_course, '② course 테이블', color='#1E8449')



── ② course 테이블 ──


,category_id,course_id,course_name
0,CAT_SN,CAT_SN_SC01,7음계
1,CAT_SN,CAT_SN_SC02,12음계
2,CAT_INT,CAT_INT_SC01,"코스1: 1도, 2도"
3,CAT_INT,CAT_INT_SC02,"코스2: 2도, 3도"
4,CAT_INT,CAT_INT_SC03,"코스3: 3도, 4도"
5,CAT_INT,CAT_INT_SC04,"코스4: 4도, 5도"
6,CAT_INT,CAT_INT_SC05,"코스5: 3도, 6도"
7,CAT_INT,CAT_INT_SC06,"코스6: 2도, 7도"


In [5]:
# ══════════════════════════════════════════════════════════════════════
# ③ part 테이블
# ══════════════════════════════════════════════════════════════════════
PART_DATA = [
    # ── CAT_SN SC01: 7음계 ──────────────────────────────────────────
    ('CAT_SN_SC01', 'CAT_SN_SC01_P01', '파트01 — C,F 구분',           ''),
    ('CAT_SN_SC01', 'CAT_SN_SC01_P02', '파트02 — C,F,G 구분',         ''),
    ('CAT_SN_SC01', 'CAT_SN_SC01_P03', '파트03 — C,D,F,G 구분',       ''),
    ('CAT_SN_SC01', 'CAT_SN_SC01_P04', '파트04 — C,D,E,F,G 구분',     ''),
    ('CAT_SN_SC01', 'CAT_SN_SC01_P05', '파트05 — C,D,E,F,G,A 구분',   ''),
    ('CAT_SN_SC01', 'CAT_SN_SC01_P06', '파트06 — C,D,E,F,G,A,B 구분', ''),
    # ── CAT_SN SC02: 12음계 ─────────────────────────────────────────
    ('CAT_SN_SC02', 'CAT_SN_SC02_P01', '파트01 — F,F# 구분',                          ''),
    ('CAT_SN_SC02', 'CAT_SN_SC02_P02', '파트02 — C,C# 구분',                          ''),
    ('CAT_SN_SC02', 'CAT_SN_SC02_P03', '파트03 — C,C#,F,F# 구분',                     ''),
    ('CAT_SN_SC02', 'CAT_SN_SC02_P04', '파트04 — G,G# 구분',                          ''),
    ('CAT_SN_SC02', 'CAT_SN_SC02_P05', '파트05 — C,C#,F,F#,G,G# 구분',               ''),
    ('CAT_SN_SC02', 'CAT_SN_SC02_P06', '파트06 — A,Bb,B 구분',                        ''),
    ('CAT_SN_SC02', 'CAT_SN_SC02_P07', '파트07 — C,C#,F,F#,G,G#,A,Bb,B 구분',       ''),
    ('CAT_SN_SC02', 'CAT_SN_SC02_P08', '파트08 — D,Eb,E 구분',                        ''),
    ('CAT_SN_SC02', 'CAT_SN_SC02_P09', '파트09 — 12음 전체 구분',                     ''),
    # ── CAT_INT SC01~SC06 ───────────────────────────────────────────
    ('CAT_INT_SC01', 'CAT_INT_SC01_P01', '파트01 — 완전1도, 장2도',        'P1,M2'),
    ('CAT_INT_SC01', 'CAT_INT_SC01_P02', '파트02 — 단2도, 장2도',          'm2,M2'),
    ('CAT_INT_SC01', 'CAT_INT_SC01_P03', '파트03 — 전체복습',              'P1,m2,M2'),
    ('CAT_INT_SC02', 'CAT_INT_SC02_P01', '파트01 — 장2도, 장3도',          'M2,M3'),
    ('CAT_INT_SC02', 'CAT_INT_SC02_P02', '파트02 — 장2도, 단3도',          'M2,m3'),
    ('CAT_INT_SC02', 'CAT_INT_SC02_P03', '파트03 — 단3도, 장3도',          'm3,M3'),
    ('CAT_INT_SC02', 'CAT_INT_SC02_P04', '파트04 — 전체복습',              'M2,m3,M3'),
    ('CAT_INT_SC03', 'CAT_INT_SC03_P01', '파트01 — 장3도, 완전4도',        'M3,P4'),
    ('CAT_INT_SC03', 'CAT_INT_SC03_P02', '파트02 — 단3도, 장3도, 완전4도', 'm3,M3,P4'),
    ('CAT_INT_SC03', 'CAT_INT_SC03_P03', '파트03 — 완전4도, 증4도',        'P4,A4'),
    ('CAT_INT_SC03', 'CAT_INT_SC03_P04', '파트04 — 전체복습',              'm3,M3,P4,A4'),
    ('CAT_INT_SC04', 'CAT_INT_SC04_P01', '파트01 — 완전4도, 완전5도',      'P4,P5'),
    ('CAT_INT_SC04', 'CAT_INT_SC04_P02', '파트02 — 증4도, 완전5도',        'A4,P5'),
    ('CAT_INT_SC04', 'CAT_INT_SC04_P03', '파트03 — 전체복습',              'P4,A4,P5'),
    ('CAT_INT_SC05', 'CAT_INT_SC05_P01', '파트01 — 장3도, 단6도',          'M3,m6'),
    ('CAT_INT_SC05', 'CAT_INT_SC05_P02', '파트02 — 단3도, 장6도',          'm3,M6'),
    ('CAT_INT_SC05', 'CAT_INT_SC05_P03', '파트03 — 단6도, 장6도',          'm6,M6'),
    ('CAT_INT_SC05', 'CAT_INT_SC05_P04', '파트04 — 전체복습',              'm3,M3,m6,M6'),
    ('CAT_INT_SC06', 'CAT_INT_SC06_P01', '파트01 — 장2도, 단7도',          'M2,m7'),
    ('CAT_INT_SC06', 'CAT_INT_SC06_P02', '파트02 — 단2도, 장7도',          'm2,M7'),
    ('CAT_INT_SC06', 'CAT_INT_SC06_P03', '파트03 — 단7도, 장7도',          'm7,M7'),
    ('CAT_INT_SC06', 'CAT_INT_SC06_P04', '파트04 — 전체복습',              'm2,M2,m7,M7'),
]
df_part = pd.DataFrame(PART_DATA, columns=['course_id','part_id','part_name','interval_pool'])
show(df_part, '③ part 테이블', color='#1E8449')



── ③ part 테이블 ──


,course_id,part_id,part_name,interval_pool
0,CAT_SN_SC01,CAT_SN_SC01_P01,"파트01 — C,F 구분",
1,CAT_SN_SC01,CAT_SN_SC01_P02,"파트02 — C,F,G 구분",
2,CAT_SN_SC01,CAT_SN_SC01_P03,"파트03 — C,D,F,G 구분",
3,CAT_SN_SC01,CAT_SN_SC01_P04,"파트04 — C,D,E,F,G 구분",
4,CAT_SN_SC01,CAT_SN_SC01_P05,"파트05 — C,D,E,F,G,A 구분",
5,CAT_SN_SC01,CAT_SN_SC01_P06,"파트06 — C,D,E,F,G,A,B 구분",
6,CAT_SN_SC02,CAT_SN_SC02_P01,"파트01 — F,F# 구분",
7,CAT_SN_SC02,CAT_SN_SC02_P02,"파트02 — C,C# 구분",
8,CAT_SN_SC02,CAT_SN_SC02_P03,"파트03 — C,C#,F,F# 구분",
9,CAT_SN_SC02,CAT_SN_SC02_P04,"파트04 — G,G# 구분",


In [6]:
# ══════════════════════════════════════════════════════════════════════
# ④ 커리큘럼(step) 테이블
# step_id: 전역 고유값. part_id, answer_type, direction 포함
# direction: '-'=단일음 / ascending/descending/harmonic=음정
# present_count: 출제 단위 수 (음정1개=1, same_diff/height_compare=2)
#   실제 재생 음 개수는 앱이 question_type 보고 계산
# ══════════════════════════════════════════════════════════════════════
CURRICULUM_DATA = [
    # part_id,             step_id,                   step_name,        question_type,   note_pool/interval_pool,                       direction,   answer_type,       difficulty_level
    # ── CAT_SN SC01 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
    ('CAT_SN_SC01_P01', 'CAT_SN_SC01_P01_S01', '같음/다름',   'single_note', 'C,F',                          '-', 'same_diff',   1),
    ('CAT_SN_SC01_P01', 'CAT_SN_SC01_P01_S02', '악보',         'single_note', 'C,F',                          '-', 'name_2choice',1),
    ('CAT_SN_SC01_P01', 'CAT_SN_SC01_P01_S03', '음이름',       'single_note', 'C,F',                          '-', 'name_2choice',1),
    ('CAT_SN_SC01_P01', 'CAT_SN_SC01_P01_S04', '피아노주관식', 'single_note', 'C,F',                          '-', 'piano_subj',  1),
    ('CAT_SN_SC01_P02', 'CAT_SN_SC01_P02_S01', '같음/다름',   'single_note', 'C,F,G',                        '-', 'same_diff',   1),
    ('CAT_SN_SC01_P02', 'CAT_SN_SC01_P02_S02', '악보',         'single_note', 'C,F,G',                        '-', 'name_2choice',1),
    ('CAT_SN_SC01_P02', 'CAT_SN_SC01_P02_S03', '음이름',       'single_note', 'C,F,G',                        '-', 'name_2choice',1),
    ('CAT_SN_SC01_P02', 'CAT_SN_SC01_P02_S04', '피아노주관식', 'single_note', 'C,F,G',                        '-', 'piano_subj',  1),
    ('CAT_SN_SC01_P03', 'CAT_SN_SC01_P03_S01', '같음/다름',   'single_note', 'C,D,F,G',                      '-', 'same_diff',   1),
    ('CAT_SN_SC01_P03', 'CAT_SN_SC01_P03_S02', '악보',         'single_note', 'C,D,F,G',                      '-', 'name_2choice',2),
    ('CAT_SN_SC01_P03', 'CAT_SN_SC01_P03_S03', '음이름',       'single_note', 'C,D,F,G',                      '-', 'name_2choice',2),
    ('CAT_SN_SC01_P03', 'CAT_SN_SC01_P03_S04', '음이름(4지)',  'single_note', 'C,D,F,G',                      '-', 'name_4choice',2),
    ('CAT_SN_SC01_P03', 'CAT_SN_SC01_P03_S05', '피아노주관식', 'single_note', 'C,D,F,G',                      '-', 'piano_subj',  2),
    ('CAT_SN_SC01_P04', 'CAT_SN_SC01_P04_S01', '같음/다름',   'single_note', 'C,D,E,F,G',                    '-', 'same_diff',   1),
    ('CAT_SN_SC01_P04', 'CAT_SN_SC01_P04_S02', '악보',         'single_note', 'C,D,E,F,G',                    '-', 'name_2choice',2),
    ('CAT_SN_SC01_P04', 'CAT_SN_SC01_P04_S03', '음이름',       'single_note', 'C,D,E,F,G',                    '-', 'name_2choice',2),
    ('CAT_SN_SC01_P04', 'CAT_SN_SC01_P04_S04', '음이름(4지)',  'single_note', 'C,D,E,F,G',                    '-', 'name_4choice',2),
    ('CAT_SN_SC01_P04', 'CAT_SN_SC01_P04_S05', '피아노주관식', 'single_note', 'C,D,E,F,G',                    '-', 'piano_subj',  2),
    ('CAT_SN_SC01_P05', 'CAT_SN_SC01_P05_S01', '같음/다름',   'single_note', 'C,D,E,F,G,A',                  '-', 'same_diff',   2),
    ('CAT_SN_SC01_P05', 'CAT_SN_SC01_P05_S02', '악보',         'single_note', 'C,D,E,F,G,A',                  '-', 'name_2choice',2),
    ('CAT_SN_SC01_P05', 'CAT_SN_SC01_P05_S03', '음이름',       'single_note', 'C,D,E,F,G,A',                  '-', 'name_2choice',2),
    ('CAT_SN_SC01_P05', 'CAT_SN_SC01_P05_S04', '음이름(4지)',  'single_note', 'C,D,E,F,G,A',                  '-', 'name_4choice',2),
    ('CAT_SN_SC01_P05', 'CAT_SN_SC01_P05_S05', '피아노주관식', 'single_note', 'C,D,E,F,G,A',                  '-', 'piano_subj',  2),
    ('CAT_SN_SC01_P06', 'CAT_SN_SC01_P06_S01', '같음/다름',   'single_note', 'C,D,E,F,G,A,B',                '-', 'same_diff',   2),
    ('CAT_SN_SC01_P06', 'CAT_SN_SC01_P06_S02', '악보',         'single_note', 'C,D,E,F,G,A,B',                '-', 'name_2choice',3),
    ('CAT_SN_SC01_P06', 'CAT_SN_SC01_P06_S03', '음이름',       'single_note', 'C,D,E,F,G,A,B',                '-', 'name_2choice',3),
    ('CAT_SN_SC01_P06', 'CAT_SN_SC01_P06_S04', '음이름(4지)',  'single_note', 'C,D,E,F,G,A,B',                '-', 'name_4choice',3),
    ('CAT_SN_SC01_P06', 'CAT_SN_SC01_P06_S05', '피아노주관식', 'single_note', 'C,D,E,F,G,A,B',                '-', 'piano_subj',  3),
    # ── CAT_SN SC02 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
    ('CAT_SN_SC02_P01', 'CAT_SN_SC02_P01_S01', '같음/다름',        'single_note', 'F,F#',                    '-', 'same_diff',   1),
    ('CAT_SN_SC02_P01', 'CAT_SN_SC02_P01_S02', '악보',              'single_note', 'F,F#',                    '-', 'name_2choice',1),
    ('CAT_SN_SC02_P01', 'CAT_SN_SC02_P01_S03', '음이름',            'single_note', 'F,F#',                    '-', 'name_2choice',1),
    ('CAT_SN_SC02_P01', 'CAT_SN_SC02_P01_S04', '피아노주관식',      'single_note', 'F,F#',                    '-', 'piano_subj',  1),
    ('CAT_SN_SC02_P02', 'CAT_SN_SC02_P02_S01', '같음/다름',        'single_note', 'C,C#',                    '-', 'same_diff',   1),
    ('CAT_SN_SC02_P02', 'CAT_SN_SC02_P02_S02', '악보',              'single_note', 'C,C#',                    '-', 'name_2choice',1),
    ('CAT_SN_SC02_P02', 'CAT_SN_SC02_P02_S03', '음이름',            'single_note', 'C,C#',                    '-', 'name_2choice',1),
    ('CAT_SN_SC02_P02', 'CAT_SN_SC02_P02_S04', '피아노주관식',      'single_note', 'C,C#',                    '-', 'piano_subj',  1),
    ('CAT_SN_SC02_P03', 'CAT_SN_SC02_P03_S01', '같음/다름',        'single_note', 'C,C#,F,F#',               '-', 'same_diff',   1),
    ('CAT_SN_SC02_P03', 'CAT_SN_SC02_P03_S02', '악보',              'single_note', 'C,C#,F,F#',               '-', 'name_2choice',2),
    ('CAT_SN_SC02_P03', 'CAT_SN_SC02_P03_S03', '음이름',            'single_note', 'C,C#,F,F#',               '-', 'name_2choice',2),
    ('CAT_SN_SC02_P03', 'CAT_SN_SC02_P03_S04', '음이름(4지)',       'single_note', 'C,C#,F,F#',               '-', 'name_4choice',2),
    ('CAT_SN_SC02_P03', 'CAT_SN_SC02_P03_S05', '피아노주관식',      'single_note', 'C,C#,F,F#',               '-', 'piano_subj',  2),
    ('CAT_SN_SC02_P04', 'CAT_SN_SC02_P04_S01', '같음/다름',        'single_note', 'G,G#',                    '-', 'same_diff',   1),
    ('CAT_SN_SC02_P04', 'CAT_SN_SC02_P04_S02', '악보',              'single_note', 'G,G#',                    '-', 'name_2choice',1),
    ('CAT_SN_SC02_P04', 'CAT_SN_SC02_P04_S03', '음이름',            'single_note', 'G,G#',                    '-', 'name_2choice',1),
    ('CAT_SN_SC02_P04', 'CAT_SN_SC02_P04_S04', '피아노주관식',      'single_note', 'G,G#',                    '-', 'piano_subj',  1),
    ('CAT_SN_SC02_P05', 'CAT_SN_SC02_P05_S01', '같음/다름',        'single_note', 'C,C#,F,F#,G,G#',          '-', 'same_diff',   2),
    ('CAT_SN_SC02_P05', 'CAT_SN_SC02_P05_S02', '악보',              'single_note', 'C,C#,F,F#,G,G#',          '-', 'name_2choice',2),
    ('CAT_SN_SC02_P05', 'CAT_SN_SC02_P05_S03', '음이름',            'single_note', 'C,C#,F,F#,G,G#',          '-', 'name_2choice',2),
    ('CAT_SN_SC02_P05', 'CAT_SN_SC02_P05_S04', '음이름(4지)',       'single_note', 'C,C#,F,F#,G,G#',          '-', 'name_4choice',2),
    ('CAT_SN_SC02_P05', 'CAT_SN_SC02_P05_S05', '피아노주관식',      'single_note', 'C,C#,F,F#,G,G#',          '-', 'piano_subj',  2),
    ('CAT_SN_SC02_P06', 'CAT_SN_SC02_P06_S01', '같음/다름',        'single_note', 'A,Bb,B',                  '-', 'same_diff',   2),
    ('CAT_SN_SC02_P06', 'CAT_SN_SC02_P06_S02', '악보',              'single_note', 'A,Bb,B',                  '-', 'name_2choice',2),
    ('CAT_SN_SC02_P06', 'CAT_SN_SC02_P06_S03', '음이름(3지)',       'single_note', 'A,Bb,B',                  '-', 'name_3choice',2),
    ('CAT_SN_SC02_P06', 'CAT_SN_SC02_P06_S04', '피아노주관식',      'single_note', 'A,Bb,B',                  '-', 'piano_subj',  2),
    ('CAT_SN_SC02_P07', 'CAT_SN_SC02_P07_S01', '같음/다름',        'single_note', 'C,C#,F,F#,G,G#,A,Bb,B',  '-', 'same_diff',   2),
    ('CAT_SN_SC02_P07', 'CAT_SN_SC02_P07_S02', '악보',              'single_note', 'C,C#,F,F#,G,G#,A,Bb,B',  '-', 'name_2choice',2),
    ('CAT_SN_SC02_P07', 'CAT_SN_SC02_P07_S03', '음이름',            'single_note', 'C,C#,F,F#,G,G#,A,Bb,B',  '-', 'name_2choice',2),
    ('CAT_SN_SC02_P07', 'CAT_SN_SC02_P07_S04', '음이름(4지)',       'single_note', 'C,C#,F,F#,G,G#,A,Bb,B',  '-', 'name_4choice',3),
    ('CAT_SN_SC02_P07', 'CAT_SN_SC02_P07_S05', '피아노주관식',      'single_note', 'C,C#,F,F#,G,G#,A,Bb,B',  '-', 'piano_subj',  3),
    ('CAT_SN_SC02_P08', 'CAT_SN_SC02_P08_S01', '같음/다름',        'single_note', 'D,Eb,E',                  '-', 'same_diff',   2),
    ('CAT_SN_SC02_P08', 'CAT_SN_SC02_P08_S02', '악보',              'single_note', 'D,Eb,E',                  '-', 'name_2choice',2),
    ('CAT_SN_SC02_P08', 'CAT_SN_SC02_P08_S03', '음이름(3지)',       'single_note', 'D,Eb,E',                  '-', 'name_3choice',2),
    ('CAT_SN_SC02_P08', 'CAT_SN_SC02_P08_S04', '피아노주관식',      'single_note', 'D,Eb,E',                  '-', 'piano_subj',  2),
    ('CAT_SN_SC02_P09', 'CAT_SN_SC02_P09_S01', '같음/다름',        'single_note', 'C,C#,D,Eb,E,F,F#,G,G#,A,Bb,B', '-', 'same_diff',   3),
    ('CAT_SN_SC02_P09', 'CAT_SN_SC02_P09_S02', '악보',              'single_note', 'C,C#,D,Eb,E,F,F#,G,G#,A,Bb,B', '-', 'name_2choice',3),
    ('CAT_SN_SC02_P09', 'CAT_SN_SC02_P09_S03', '음이름',            'single_note', 'C,C#,D,Eb,E,F,F#,G,G#,A,Bb,B', '-', 'name_2choice',3),
    ('CAT_SN_SC02_P09', 'CAT_SN_SC02_P09_S04', '음이름(4지)',       'single_note', 'C,C#,D,Eb,E,F,F#,G,G#,A,Bb,B', '-', 'name_4choice',3),
    ('CAT_SN_SC02_P09', 'CAT_SN_SC02_P09_S05', '피아노주관식',      'single_note', 'C,C#,D,Eb,E,F,F#,G,G#,A,Bb,B', '-', 'piano_subj',  3),
    # ── CAT_INT ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
    ('CAT_INT_SC01_P01', 'CAT_INT_SC01_P01_S01', '음정 같음/다름', 'interval', 'P1,M2', 'ascending', 'same_diff', 1),
    ('CAT_INT_SC01_P01', 'CAT_INT_SC01_P01_S02', '다양한 높이 비교', 'interval', 'P1,M2', 'ascending', 'height_compare', 1),
    ('CAT_INT_SC01_P01', 'CAT_INT_SC01_P01_S03', '음정 이름 고르기', 'interval', 'P1,M2', 'ascending', 'name_2choice', 1),
    ('CAT_INT_SC01_P01', 'CAT_INT_SC01_P01_S04', '상행 음정 알아맞히기', 'interval', 'P1,M2', 'ascending', 'interval_subj', 1),
    ('CAT_INT_SC01_P01', 'CAT_INT_SC01_P01_S05', '하행 음정 알아맞히기', 'interval', 'P1,M2', 'descending', 'interval_subj', 1),
    ('CAT_INT_SC01_P01', 'CAT_INT_SC01_P01_S06', '건반에서 음정 선택', 'interval', 'P1,M2', 'ascending', 'keyboard_subj', 1),
    ('CAT_INT_SC01_P01', 'CAT_INT_SC01_P01_S07', '화음에서 음정 찾기', 'interval', 'P1,M2', 'harmonic', 'interval_subj', 1),
    ('CAT_INT_SC01_P02', 'CAT_INT_SC01_P02_S01', '음정 같음/다름', 'interval', 'm2,M2', 'ascending', 'same_diff', 1),
    ('CAT_INT_SC01_P02', 'CAT_INT_SC01_P02_S02', '다양한 높이 비교', 'interval', 'm2,M2', 'ascending', 'height_compare', 1),
    ('CAT_INT_SC01_P02', 'CAT_INT_SC01_P02_S03', '음정 이름 고르기', 'interval', 'm2,M2', 'ascending', 'name_2choice', 1),
    ('CAT_INT_SC01_P02', 'CAT_INT_SC01_P02_S04', '상행 음정 알아맞히기', 'interval', 'm2,M2', 'ascending', 'interval_subj', 1),
    ('CAT_INT_SC01_P02', 'CAT_INT_SC01_P02_S05', '하행 음정 알아맞히기', 'interval', 'm2,M2', 'descending', 'interval_subj', 1),
    ('CAT_INT_SC01_P02', 'CAT_INT_SC01_P02_S06', '건반에서 음정 선택', 'interval', 'm2,M2', 'ascending', 'keyboard_subj', 1),
    ('CAT_INT_SC01_P02', 'CAT_INT_SC01_P02_S07', '화음에서 음정 찾기', 'interval', 'm2,M2', 'harmonic', 'interval_subj', 1),
    ('CAT_INT_SC01_P03', 'CAT_INT_SC01_P03_S01', '음정 이름 고르기', 'interval', 'P1,m2,M2', 'ascending', 'name_3choice', 2),
    ('CAT_INT_SC01_P03', 'CAT_INT_SC01_P03_S02', '상행 음정 알아맞히기', 'interval', 'P1,m2,M2', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC01_P03', 'CAT_INT_SC01_P03_S03', '하행 음정 알아맞히기', 'interval', 'P1,m2,M2', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC01_P03', 'CAT_INT_SC01_P03_S04', '건반에서 음정 선택', 'interval', 'P1,m2,M2', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC01_P03', 'CAT_INT_SC01_P03_S05', '화음에서 음정 찾기', 'interval', 'P1,m2,M2', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC02_P01', 'CAT_INT_SC02_P01_S01', '음정 같음/다름', 'interval', 'M2,M3', 'ascending', 'same_diff', 1),
    ('CAT_INT_SC02_P01', 'CAT_INT_SC02_P01_S02', '다양한 높이 비교', 'interval', 'M2,M3', 'ascending', 'height_compare', 1),
    ('CAT_INT_SC02_P01', 'CAT_INT_SC02_P01_S03', '음정 이름 고르기', 'interval', 'M2,M3', 'ascending', 'name_2choice', 1),
    ('CAT_INT_SC02_P01', 'CAT_INT_SC02_P01_S04', '상행 음정 알아맞히기', 'interval', 'M2,M3', 'ascending', 'interval_subj', 1),
    ('CAT_INT_SC02_P01', 'CAT_INT_SC02_P01_S05', '하행 음정 알아맞히기', 'interval', 'M2,M3', 'descending', 'interval_subj', 1),
    ('CAT_INT_SC02_P01', 'CAT_INT_SC02_P01_S06', '건반에서 음정 선택', 'interval', 'M2,M3', 'ascending', 'keyboard_subj', 1),
    ('CAT_INT_SC02_P01', 'CAT_INT_SC02_P01_S07', '화음에서 음정 찾기', 'interval', 'M2,M3', 'harmonic', 'interval_subj', 1),
    ('CAT_INT_SC02_P02', 'CAT_INT_SC02_P02_S01', '음정 같음/다름', 'interval', 'M2,m3', 'ascending', 'same_diff', 1),
    ('CAT_INT_SC02_P02', 'CAT_INT_SC02_P02_S02', '다양한 높이 비교', 'interval', 'M2,m3', 'ascending', 'height_compare', 1),
    ('CAT_INT_SC02_P02', 'CAT_INT_SC02_P02_S03', '음정 이름 고르기', 'interval', 'M2,m3', 'ascending', 'name_2choice', 1),
    ('CAT_INT_SC02_P02', 'CAT_INT_SC02_P02_S04', '상행 음정 알아맞히기', 'interval', 'M2,m3', 'ascending', 'interval_subj', 1),
    ('CAT_INT_SC02_P02', 'CAT_INT_SC02_P02_S05', '하행 음정 알아맞히기', 'interval', 'M2,m3', 'descending', 'interval_subj', 1),
    ('CAT_INT_SC02_P02', 'CAT_INT_SC02_P02_S06', '건반에서 음정 선택', 'interval', 'M2,m3', 'ascending', 'keyboard_subj', 1),
    ('CAT_INT_SC02_P02', 'CAT_INT_SC02_P02_S07', '화음에서 음정 찾기', 'interval', 'M2,m3', 'harmonic', 'interval_subj', 1),
    ('CAT_INT_SC02_P03', 'CAT_INT_SC02_P03_S01', '음정 같음/다름', 'interval', 'm3,M3', 'ascending', 'same_diff', 1),
    ('CAT_INT_SC02_P03', 'CAT_INT_SC02_P03_S02', '다양한 높이 비교', 'interval', 'm3,M3', 'ascending', 'height_compare', 1),
    ('CAT_INT_SC02_P03', 'CAT_INT_SC02_P03_S03', '음정 이름 고르기', 'interval', 'm3,M3', 'ascending', 'name_2choice', 1),
    ('CAT_INT_SC02_P03', 'CAT_INT_SC02_P03_S04', '상행 음정 알아맞히기', 'interval', 'm3,M3', 'ascending', 'interval_subj', 1),
    ('CAT_INT_SC02_P03', 'CAT_INT_SC02_P03_S05', '하행 음정 알아맞히기', 'interval', 'm3,M3', 'descending', 'interval_subj', 1),
    ('CAT_INT_SC02_P03', 'CAT_INT_SC02_P03_S06', '건반에서 음정 선택', 'interval', 'm3,M3', 'ascending', 'keyboard_subj', 1),
    ('CAT_INT_SC02_P03', 'CAT_INT_SC02_P03_S07', '화음에서 음정 찾기', 'interval', 'm3,M3', 'harmonic', 'interval_subj', 1),
    ('CAT_INT_SC02_P04', 'CAT_INT_SC02_P04_S01', '음정 이름 고르기', 'interval', 'M2,m3,M3', 'ascending', 'name_3choice', 2),
    ('CAT_INT_SC02_P04', 'CAT_INT_SC02_P04_S02', '상행 음정 알아맞히기', 'interval', 'M2,m3,M3', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC02_P04', 'CAT_INT_SC02_P04_S03', '하행 음정 알아맞히기', 'interval', 'M2,m3,M3', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC02_P04', 'CAT_INT_SC02_P04_S04', '건반에서 음정 선택', 'interval', 'M2,m3,M3', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC02_P04', 'CAT_INT_SC02_P04_S05', '화음에서 음정 찾기', 'interval', 'M2,m3,M3', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC03_P01', 'CAT_INT_SC03_P01_S01', '음정 같음/다름', 'interval', 'M3,P4', 'ascending', 'same_diff', 1),
    ('CAT_INT_SC03_P01', 'CAT_INT_SC03_P01_S02', '다양한 높이 비교', 'interval', 'M3,P4', 'ascending', 'height_compare', 1),
    ('CAT_INT_SC03_P01', 'CAT_INT_SC03_P01_S03', '음정 이름 고르기', 'interval', 'M3,P4', 'ascending', 'name_2choice', 1),
    ('CAT_INT_SC03_P01', 'CAT_INT_SC03_P01_S04', '상행 음정 알아맞히기', 'interval', 'M3,P4', 'ascending', 'interval_subj', 1),
    ('CAT_INT_SC03_P01', 'CAT_INT_SC03_P01_S05', '하행 음정 알아맞히기', 'interval', 'M3,P4', 'descending', 'interval_subj', 1),
    ('CAT_INT_SC03_P01', 'CAT_INT_SC03_P01_S06', '건반에서 음정 선택', 'interval', 'M3,P4', 'ascending', 'keyboard_subj', 1),
    ('CAT_INT_SC03_P01', 'CAT_INT_SC03_P01_S07', '화음에서 음정 찾기', 'interval', 'M3,P4', 'harmonic', 'interval_subj', 1),
    ('CAT_INT_SC03_P02', 'CAT_INT_SC03_P02_S01', '음정 같음/다름', 'interval', 'm3,M3,P4', 'ascending', 'same_diff', 2),
    ('CAT_INT_SC03_P02', 'CAT_INT_SC03_P02_S02', '다양한 높이 비교', 'interval', 'm3,M3,P4', 'ascending', 'height_compare', 2),
    ('CAT_INT_SC03_P02', 'CAT_INT_SC03_P02_S03', '음정 이름 고르기', 'interval', 'm3,M3,P4', 'ascending', 'name_3choice', 2),
    ('CAT_INT_SC03_P02', 'CAT_INT_SC03_P02_S04', '상행 음정 알아맞히기', 'interval', 'm3,M3,P4', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC03_P02', 'CAT_INT_SC03_P02_S05', '하행 음정 알아맞히기', 'interval', 'm3,M3,P4', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC03_P02', 'CAT_INT_SC03_P02_S06', '건반에서 음정 선택', 'interval', 'm3,M3,P4', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC03_P02', 'CAT_INT_SC03_P02_S07', '화음에서 음정 찾기', 'interval', 'm3,M3,P4', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC03_P03', 'CAT_INT_SC03_P03_S01', '음정 같음/다름', 'interval', 'P4,A4', 'ascending', 'same_diff', 2),
    ('CAT_INT_SC03_P03', 'CAT_INT_SC03_P03_S02', '다양한 높이 비교', 'interval', 'P4,A4', 'ascending', 'height_compare', 2),
    ('CAT_INT_SC03_P03', 'CAT_INT_SC03_P03_S03', '음정 이름 고르기', 'interval', 'P4,A4', 'ascending', 'name_2choice', 2),
    ('CAT_INT_SC03_P03', 'CAT_INT_SC03_P03_S04', '상행 음정 알아맞히기', 'interval', 'P4,A4', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC03_P03', 'CAT_INT_SC03_P03_S05', '하행 음정 알아맞히기', 'interval', 'P4,A4', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC03_P03', 'CAT_INT_SC03_P03_S06', '건반에서 음정 선택', 'interval', 'P4,A4', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC03_P03', 'CAT_INT_SC03_P03_S07', '화음에서 음정 찾기', 'interval', 'P4,A4', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC03_P04', 'CAT_INT_SC03_P04_S01', '음정 이름 고르기', 'interval', 'm3,M3,P4,A4', 'ascending', 'name_4choice', 2),
    ('CAT_INT_SC03_P04', 'CAT_INT_SC03_P04_S02', '상행 음정 알아맞히기', 'interval', 'm3,M3,P4,A4', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC03_P04', 'CAT_INT_SC03_P04_S03', '하행 음정 알아맞히기', 'interval', 'm3,M3,P4,A4', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC03_P04', 'CAT_INT_SC03_P04_S04', '건반에서 음정 선택', 'interval', 'm3,M3,P4,A4', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC03_P04', 'CAT_INT_SC03_P04_S05', '화음에서 음정 찾기', 'interval', 'm3,M3,P4,A4', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC04_P01', 'CAT_INT_SC04_P01_S01', '음정 같음/다름', 'interval', 'P4,P5', 'ascending', 'same_diff', 2),
    ('CAT_INT_SC04_P01', 'CAT_INT_SC04_P01_S02', '다양한 높이 비교', 'interval', 'P4,P5', 'ascending', 'height_compare', 2),
    ('CAT_INT_SC04_P01', 'CAT_INT_SC04_P01_S03', '음정 이름 고르기', 'interval', 'P4,P5', 'ascending', 'name_2choice', 2),
    ('CAT_INT_SC04_P01', 'CAT_INT_SC04_P01_S04', '상행 음정 알아맞히기', 'interval', 'P4,P5', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC04_P01', 'CAT_INT_SC04_P01_S05', '하행 음정 알아맞히기', 'interval', 'P4,P5', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC04_P01', 'CAT_INT_SC04_P01_S06', '건반에서 음정 선택', 'interval', 'P4,P5', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC04_P01', 'CAT_INT_SC04_P01_S07', '화음에서 음정 찾기', 'interval', 'P4,P5', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC04_P02', 'CAT_INT_SC04_P02_S01', '음정 같음/다름', 'interval', 'A4,P5', 'ascending', 'same_diff', 2),
    ('CAT_INT_SC04_P02', 'CAT_INT_SC04_P02_S02', '다양한 높이 비교', 'interval', 'A4,P5', 'ascending', 'height_compare', 2),
    ('CAT_INT_SC04_P02', 'CAT_INT_SC04_P02_S03', '음정 이름 고르기', 'interval', 'A4,P5', 'ascending', 'name_2choice', 2),
    ('CAT_INT_SC04_P02', 'CAT_INT_SC04_P02_S04', '상행 음정 알아맞히기', 'interval', 'A4,P5', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC04_P02', 'CAT_INT_SC04_P02_S05', '하행 음정 알아맞히기', 'interval', 'A4,P5', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC04_P02', 'CAT_INT_SC04_P02_S06', '건반에서 음정 선택', 'interval', 'A4,P5', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC04_P02', 'CAT_INT_SC04_P02_S07', '화음에서 음정 찾기', 'interval', 'A4,P5', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC04_P03', 'CAT_INT_SC04_P03_S01', '음정 이름 고르기', 'interval', 'P4,A4,P5', 'ascending', 'name_3choice', 2),
    ('CAT_INT_SC04_P03', 'CAT_INT_SC04_P03_S02', '상행 음정 알아맞히기', 'interval', 'P4,A4,P5', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC04_P03', 'CAT_INT_SC04_P03_S03', '하행 음정 알아맞히기', 'interval', 'P4,A4,P5', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC04_P03', 'CAT_INT_SC04_P03_S04', '건반에서 음정 선택', 'interval', 'P4,A4,P5', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC04_P03', 'CAT_INT_SC04_P03_S05', '화음에서 음정 찾기', 'interval', 'P4,A4,P5', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC05_P01', 'CAT_INT_SC05_P01_S01', '음정 같음/다름', 'interval', 'M3,m6', 'ascending', 'same_diff', 2),
    ('CAT_INT_SC05_P01', 'CAT_INT_SC05_P01_S02', '다양한 높이 비교', 'interval', 'M3,m6', 'ascending', 'height_compare', 2),
    ('CAT_INT_SC05_P01', 'CAT_INT_SC05_P01_S03', '음정 이름 고르기', 'interval', 'M3,m6', 'ascending', 'name_2choice', 2),
    ('CAT_INT_SC05_P01', 'CAT_INT_SC05_P01_S04', '상행 음정 알아맞히기', 'interval', 'M3,m6', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC05_P01', 'CAT_INT_SC05_P01_S05', '하행 음정 알아맞히기', 'interval', 'M3,m6', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC05_P01', 'CAT_INT_SC05_P01_S06', '건반에서 음정 선택', 'interval', 'M3,m6', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC05_P01', 'CAT_INT_SC05_P01_S07', '화음에서 음정 찾기', 'interval', 'M3,m6', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC05_P02', 'CAT_INT_SC05_P02_S01', '음정 같음/다름', 'interval', 'm3,M6', 'ascending', 'same_diff', 2),
    ('CAT_INT_SC05_P02', 'CAT_INT_SC05_P02_S02', '다양한 높이 비교', 'interval', 'm3,M6', 'ascending', 'height_compare', 2),
    ('CAT_INT_SC05_P02', 'CAT_INT_SC05_P02_S03', '음정 이름 고르기', 'interval', 'm3,M6', 'ascending', 'name_2choice', 2),
    ('CAT_INT_SC05_P02', 'CAT_INT_SC05_P02_S04', '상행 음정 알아맞히기', 'interval', 'm3,M6', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC05_P02', 'CAT_INT_SC05_P02_S05', '하행 음정 알아맞히기', 'interval', 'm3,M6', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC05_P02', 'CAT_INT_SC05_P02_S06', '건반에서 음정 선택', 'interval', 'm3,M6', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC05_P02', 'CAT_INT_SC05_P02_S07', '화음에서 음정 찾기', 'interval', 'm3,M6', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC05_P03', 'CAT_INT_SC05_P03_S01', '음정 같음/다름', 'interval', 'm6,M6', 'ascending', 'same_diff', 2),
    ('CAT_INT_SC05_P03', 'CAT_INT_SC05_P03_S02', '다양한 높이 비교', 'interval', 'm6,M6', 'ascending', 'height_compare', 2),
    ('CAT_INT_SC05_P03', 'CAT_INT_SC05_P03_S03', '음정 이름 고르기', 'interval', 'm6,M6', 'ascending', 'name_2choice', 2),
    ('CAT_INT_SC05_P03', 'CAT_INT_SC05_P03_S04', '상행 음정 알아맞히기', 'interval', 'm6,M6', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC05_P03', 'CAT_INT_SC05_P03_S05', '하행 음정 알아맞히기', 'interval', 'm6,M6', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC05_P03', 'CAT_INT_SC05_P03_S06', '건반에서 음정 선택', 'interval', 'm6,M6', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC05_P03', 'CAT_INT_SC05_P03_S07', '화음에서 음정 찾기', 'interval', 'm6,M6', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC05_P04', 'CAT_INT_SC05_P04_S01', '음정 이름 고르기', 'interval', 'm3,M3,m6,M6', 'ascending', 'name_4choice', 3),
    ('CAT_INT_SC05_P04', 'CAT_INT_SC05_P04_S02', '상행 음정 알아맞히기', 'interval', 'm3,M3,m6,M6', 'ascending', 'interval_subj', 3),
    ('CAT_INT_SC05_P04', 'CAT_INT_SC05_P04_S03', '하행 음정 알아맞히기', 'interval', 'm3,M3,m6,M6', 'descending', 'interval_subj', 3),
    ('CAT_INT_SC05_P04', 'CAT_INT_SC05_P04_S04', '건반에서 음정 선택', 'interval', 'm3,M3,m6,M6', 'ascending', 'keyboard_subj', 3),
    ('CAT_INT_SC05_P04', 'CAT_INT_SC05_P04_S05', '화음에서 음정 찾기', 'interval', 'm3,M3,m6,M6', 'harmonic', 'interval_subj', 3),
    ('CAT_INT_SC06_P01', 'CAT_INT_SC06_P01_S01', '음정 같음/다름', 'interval', 'M2,m7', 'ascending', 'same_diff', 2),
    ('CAT_INT_SC06_P01', 'CAT_INT_SC06_P01_S02', '다양한 높이 비교', 'interval', 'M2,m7', 'ascending', 'height_compare', 2),
    ('CAT_INT_SC06_P01', 'CAT_INT_SC06_P01_S03', '음정 이름 고르기', 'interval', 'M2,m7', 'ascending', 'name_2choice', 2),
    ('CAT_INT_SC06_P01', 'CAT_INT_SC06_P01_S04', '상행 음정 알아맞히기', 'interval', 'M2,m7', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC06_P01', 'CAT_INT_SC06_P01_S05', '하행 음정 알아맞히기', 'interval', 'M2,m7', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC06_P01', 'CAT_INT_SC06_P01_S06', '건반에서 음정 선택', 'interval', 'M2,m7', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC06_P01', 'CAT_INT_SC06_P01_S07', '화음에서 음정 찾기', 'interval', 'M2,m7', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC06_P02', 'CAT_INT_SC06_P02_S01', '음정 같음/다름', 'interval', 'm2,M7', 'ascending', 'same_diff', 2),
    ('CAT_INT_SC06_P02', 'CAT_INT_SC06_P02_S02', '다양한 높이 비교', 'interval', 'm2,M7', 'ascending', 'height_compare', 2),
    ('CAT_INT_SC06_P02', 'CAT_INT_SC06_P02_S03', '음정 이름 고르기', 'interval', 'm2,M7', 'ascending', 'name_2choice', 2),
    ('CAT_INT_SC06_P02', 'CAT_INT_SC06_P02_S04', '상행 음정 알아맞히기', 'interval', 'm2,M7', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC06_P02', 'CAT_INT_SC06_P02_S05', '하행 음정 알아맞히기', 'interval', 'm2,M7', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC06_P02', 'CAT_INT_SC06_P02_S06', '건반에서 음정 선택', 'interval', 'm2,M7', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC06_P02', 'CAT_INT_SC06_P02_S07', '화음에서 음정 찾기', 'interval', 'm2,M7', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC06_P03', 'CAT_INT_SC06_P03_S01', '음정 같음/다름', 'interval', 'm7,M7', 'ascending', 'same_diff', 2),
    ('CAT_INT_SC06_P03', 'CAT_INT_SC06_P03_S02', '다양한 높이 비교', 'interval', 'm7,M7', 'ascending', 'height_compare', 2),
    ('CAT_INT_SC06_P03', 'CAT_INT_SC06_P03_S03', '음정 이름 고르기', 'interval', 'm7,M7', 'ascending', 'name_2choice', 2),
    ('CAT_INT_SC06_P03', 'CAT_INT_SC06_P03_S04', '상행 음정 알아맞히기', 'interval', 'm7,M7', 'ascending', 'interval_subj', 2),
    ('CAT_INT_SC06_P03', 'CAT_INT_SC06_P03_S05', '하행 음정 알아맞히기', 'interval', 'm7,M7', 'descending', 'interval_subj', 2),
    ('CAT_INT_SC06_P03', 'CAT_INT_SC06_P03_S06', '건반에서 음정 선택', 'interval', 'm7,M7', 'ascending', 'keyboard_subj', 2),
    ('CAT_INT_SC06_P03', 'CAT_INT_SC06_P03_S07', '화음에서 음정 찾기', 'interval', 'm7,M7', 'harmonic', 'interval_subj', 2),
    ('CAT_INT_SC06_P04', 'CAT_INT_SC06_P04_S01', '음정 이름 고르기', 'interval', 'm2,M2,m7,M7', 'ascending', 'name_4choice', 3),
    ('CAT_INT_SC06_P04', 'CAT_INT_SC06_P04_S02', '상행 음정 알아맞히기', 'interval', 'm2,M2,m7,M7', 'ascending', 'interval_subj', 3),
    ('CAT_INT_SC06_P04', 'CAT_INT_SC06_P04_S03', '하행 음정 알아맞히기', 'interval', 'm2,M2,m7,M7', 'descending', 'interval_subj', 3),
    ('CAT_INT_SC06_P04', 'CAT_INT_SC06_P04_S04', '건반에서 음정 선택', 'interval', 'm2,M2,m7,M7', 'ascending', 'keyboard_subj', 3),
    ('CAT_INT_SC06_P04', 'CAT_INT_SC06_P04_S05', '화음에서 음정 찾기', 'interval', 'm2,M2,m7,M7', 'harmonic', 'interval_subj', 3),
]

# 컬럼 정의 (direction 컬럼 추가 — v8)
CURRICULUM_COLS = [
    'part_id', 'step_id', 'step_name', 'question_type',
    'note_pool', 'direction', 'answer_type', 'difficulty_level'
]
df_curriculum = pd.DataFrame(CURRICULUM_DATA, columns=CURRICULUM_COLS)

# O(1) 조회 dict
STEP_LOOKUP: dict[str, dict] = {
    r['step_id']: r for _, r in df_curriculum.iterrows()
}

sn_cnt  = sum(1 for r in CURRICULUM_DATA if r[3] == 'single_note')
int_cnt = sum(1 for r in CURRICULUM_DATA if r[3] == 'interval')
print(f'✅ 커리큘럼 로드 완료 — 단일음: {sn_cnt}개 / 음정: {int_cnt}개 / 합계: {sn_cnt+int_cnt}개')
ids = [r[1] for r in CURRICULUM_DATA]
dupes = set(x for x in ids if ids.count(x)>1)
print(f'   step_id 중복: {dupes if dupes else "없음 ✅"}')


✅ 커리큘럼 로드 완료 — 단일음: 68개 / 음정: 142개 / 합계: 210개
   step_id 중복: 없음 ✅


In [7]:
# ══════════════════════════════════════════════════════════════════════
# ⑤ answer_type 규칙 사전 — v8 통합 (단일음·음정 공유)
# 공유: same_diff, name_2/3/4choice
# 음정 전용: height_compare, interval_subj, keyboard_subj
# 단일음 전용: piano_subj
# ══════════════════════════════════════════════════════════════════════
ANSWER_TYPE_RULES: dict = {
    # answer_type         label              num_choices  present_count  distractor_strategy   pool_size_rule
    # ── 공유 ──────────────────────────────────────────────────────────
    'same_diff':         ('같음/다름',            2,           2,         'none',              '풀 크기 무관'),
    'name_2choice':      ('이름 2지선다',          2,           1,         'asc_by_distance',   '풀≤2:use_all / 풀≥3:asc_by_dist'),
    'name_3choice':      ('이름 3지선다',          3,           1,         'use_all',           '풀=3 전용'),
    'name_4choice':      ('이름 4지선다',          4,           1,         'asc_by_distance',   '풀≥4 전용'),
    # ── 음정 전용 ─────────────────────────────────────────────────────
    'height_compare':    ('다양한 높이 비교',      2,           2,         'none',              '음정: 같은 음정 다른 높이'),
    'interval_subj':     ('음정 주관식',           0,           1,         'none',              '음정: direction 컬럼으로 상행/하행/화음 구분'),
    'keyboard_subj':     ('건반 선택',             0,           1,         'none',              '음정: 음정이름 제시→건반 선택'),
    # ── 단일음 전용 ───────────────────────────────────────────────────
    'piano_subj':        ('피아노 주관식',         0,           1,         'none',              '단일음: 피아노 건반 직접 입력'),
}

df_answer_rules = pd.DataFrame(
    [(k,) + v for k, v in ANSWER_TYPE_RULES.items()],
    columns=['answer_type','label','num_choices','present_count','distractor_strategy','pool_size_rule']
)
show(df_answer_rules, '⑤ answer_type 규칙 사전 (v8 통합)', color='#2980B9')



── ⑤ answer_type 규칙 사전 (v8 통합) ──


,answer_type,label,num_choices,present_count,distractor_strategy,pool_size_rule
0,same_diff,같음/다름,2,2,none,풀 크기 무관
1,name_2choice,이름 2지선다,2,1,asc_by_distance,풀≤2:use_all / 풀≥3:asc_by_dist
2,name_3choice,이름 3지선다,3,1,use_all,풀=3 전용
3,name_4choice,이름 4지선다,4,1,asc_by_distance,풀≥4 전용
4,height_compare,다양한 높이 비교,2,2,none,음정: 같은 음정 다른 높이
5,interval_subj,음정 주관식,0,1,none,음정: direction 컬럼으로 상행/하행/화음 구분
6,keyboard_subj,건반 선택,0,1,none,음정: 음정이름 제시→건반 선택
7,piano_subj,피아노 주관식,0,1,none,단일음: 피아노 건반 직접 입력


In [8]:
# ══════════════════════════════════════════════════════════════════════
# ⑥ difficulty_level 규칙 사전 — v8 완전 통합 (단일음·음정 공유)
# oct_min/oct_max : 기준음 옥타브 범위
# proximity_strategy : 오답 배치 전략 (단일음=반음거리 / 음정=음정크기 차이)
# ══════════════════════════════════════════════════════════════════════
DIFFICULTY_RULES: dict = {
    1: ('쉬움',   4, 4, 'desc_by_distance', '≥5반음/≥4반음차(오답 멀리)'),
    2: ('보통',   3, 5, 'shuffle',           '무작위 배치'),
    3: ('어려움', 3, 6, 'asc_by_distance',  '≤2반음/≤1반음차(오답 가까이)'),
}
df_difficulty = pd.DataFrame(
    [(k,) + v for k, v in DIFFICULTY_RULES.items()],
    columns=['difficulty_level','label','oct_min','oct_max','proximity_strategy','proximity_semitones']
)
show(df_difficulty, '⑥ difficulty_level 규칙 사전 (v8 통합)', color='#D35400')



── ⑥ difficulty_level 규칙 사전 (v8 통합) ──


,difficulty_level,label,oct_min,oct_max,proximity_strategy,proximity_semitones
0,1,쉬움,4,4,desc_by_distance,≥5반음/≥4반음차(오답 멀리)
1,2,보통,3,5,shuffle,무작위 배치
2,3,어려움,3,6,asc_by_distance,≤2반음/≤1반음차(오답 가까이)


---
## 3. 음 유틸 (MIDI ↔ 음이름 변환)

> 음 관련 모든 계산은 내부적으로 **MIDI 번호**로 처리합니다.  
> 표기(C#, Bb 등)는 note_pool 원본 문자열을 그대로 유지합니다.  
>
> 음정 청음 단계에서 b/# 표기 기준을 통일할 때  
> `ENHARMONIC_MAP` 과 `NOTE_NAMES` 만 수정하면 나머지 로직은 그대로입니다.
>
> **MIDI 번호 기준**: `C4 = 60` / `C3 = 48` / `C6 = 84`

In [9]:
# 반음 순서 기준 표기
NOTE_NAMES: list[str] = ['C','C#','D','Eb','E','F','F#','G','G#','A','Bb','B']

# 이명동음 정규화 테이블
# 음정 청음 단계에서 b/# 기준 통일 시 이 테이블을 수정
ENHARMONIC_MAP: dict[str, str] = {
    'Db': 'C#', 'D#': 'Eb', 'Gb': 'F#',
    'Ab': 'G#', 'A#': 'Bb', 'Cb': 'B',  'E#': 'F',
}

def normalize_note_name(name: str) -> str:
    """이명동음 정규화: 'Db' → 'C#' 등 내부 표기로 통일"""
    return ENHARMONIC_MAP.get(name, name)

def note_to_midi(note_with_octave: str) -> int:
    """
    음이름+옥타브 → MIDI 번호
    예: 'C4' → 60,  'F#3' → 54,  'Bb4' → 70
    """
    octave      = int(note_with_octave[-1])
    name        = normalize_note_name(note_with_octave[:-1])
    pitch_class = NOTE_NAMES.index(name)
    return (octave + 1) * 12 + pitch_class

def midi_to_note(midi: int) -> str:
    """
    MIDI 번호 → 음이름+옥타브
    예: 60 → 'C4',  70 → 'Bb4'
    """
    return f"{NOTE_NAMES[midi % 12]}{(midi // 12) - 1}"

def parse_pool(pool_str: str) -> list[str]:
    """note_pool 문자열 → 정규화된 음이름 리스트"""
    return [normalize_note_name(n.strip()) for n in pool_str.split(',')]

def pool_to_midi_range(pool: list[str], oct_min: int, oct_max: int) -> list[int]:
    """
    pool 음이름 + 옥타브 범위 → 유효한 MIDI 번호 리스트
    전역 MIDI 범위(MIDI_MIN ~ MIDI_MAX)와 교차 적용
    """
    return sorted(
        note_to_midi(f'{name}{oct}')
        for oct  in range(oct_min, oct_max + 1)
        for name in pool
        if MIDI_MIN <= note_to_midi(f'{name}{oct}') <= MIDI_MAX
    )

def semitone_distance(midi_a: int, midi_b: int) -> int:
    """두 MIDI 번호 사이의 반음 거리 (절댓값)"""
    return abs(midi_a - midi_b)

# ── 변환 확인 ────────────────────────────────────────────────────────
print('[MIDI 변환 확인]')
for note in ['C4', 'F#3', 'Bb4', 'G#5', 'Eb4']:
    m = note_to_midi(note)
    print(f'  {note:5s} → MIDI {m:3d} → {midi_to_note(m)}')

[MIDI 변환 확인]
  C4    → MIDI  60 → C4
  F#3   → MIDI  54 → F#3
  Bb4   → MIDI  70 → Bb4
  G#5   → MIDI  80 → G#5
  Eb4   → MIDI  63 → Eb4


### 3-1. 음정 유틸 (음정 ↔ 거리를 숫자로 변환환)

> 음정 관련 모든 계산은 내부적으로 숫자로 처리합니다. (음정 사이 반음 개수)
> 표기(P1, m6 등)는 note_pool/interval_pool 원본 문자열을 그대로 유지합니다.  

In [11]:
# ══════════════════════════════════════════════════════════════════════
# 음정 유틸 — CAT_INT 전용
# ══════════════════════════════════════════════════════════════════════

# 기호 → (한국어명, 반음수)
INTERVAL_SEMITONES: dict = {
    'P1': ('완전1도',  0),
    'm2': ('단2도',    1),
    'M2': ('장2도',    2),
    'm3': ('단3도',    3),
    'M3': ('장3도',    4),
    'P4': ('완전4도',  5),
    'A4': ('증4도',    6),
    'P5': ('완전5도',  7),
    'm6': ('단6도',    8),
    'M6': ('장6도',    9),
    'm7': ('단7도',   10),
    'M7': ('장7도',   11),
}
# 역방향: 반음수 → 기호
SEMITONES_TO_INTERVAL: dict = {v[1]: k for k, v in INTERVAL_SEMITONES.items()}

def parse_interval_pool(pool_str: str) -> list:
    # interval_pool 문자열 → 기호 리스트. 예: 'P1,M2' → ['P1','M2']
    return [s.strip() for s in pool_str.split(',')]

def interval_semitones(symbol: str) -> int:
    # 음정 기호 → 반음수. 예: 'M3' → 4
    if symbol not in INTERVAL_SEMITONES:
        raise ValueError(f"알 수 없는 음정 기호: '{symbol}'")
    return INTERVAL_SEMITONES[symbol][1]

def interval_name_ko(symbol: str) -> str:
    # 음정 기호 → 한국어 이름. 예: 'M3' → '장3도'
    return INTERVAL_SEMITONES[symbol][0]

def build_interval_midi(root_midi: int, symbol: str, direction: str) -> int:
    # 기준음 + 음정 + 방향 → 상단음(또는 하단음) MIDI
    # direction='ascending'  → root + semitones
    # direction='descending' → root - semitones
    # direction='harmonic'   → root + semitones (동시 재생이므로 상단음 반환)
    st = interval_semitones(symbol)
    if direction == 'descending':
        return root_midi - st
    return root_midi + st

def interval_to_midi_pair(root_midi: int, symbol: str, direction: str) -> tuple:
    # 기준음 + 음정 → (낮은음 MIDI, 높은음 MIDI) 쌍
    # direction='descending' 이면 root가 높은음
    st = interval_semitones(symbol)
    if direction == 'descending':
        return (root_midi - st, root_midi)
    return (root_midi, root_midi + st)

def root_midi_pool(oct_min: int, oct_max: int) -> list:
    # 기준음 MIDI 풀 — 모든 12음 × 옥타브 범위
    return [
        note_to_midi(f'{n}{o}')
        for o in range(oct_min, oct_max + 1)
        for n in NOTE_NAMES
        if MIDI_MIN <= note_to_midi(f'{n}{o}') <= MIDI_MAX
    ]

print('✅ 음정 유틸 정의 완료')
print(f'   음정 기호 {len(INTERVAL_SEMITONES)}개: {list(INTERVAL_SEMITONES.keys())}')


✅ 음정 유틸 정의 완료
   음정 기호 12개: ['P1', 'm2', 'M2', 'm3', 'M3', 'P4', 'A4', 'P5', 'm6', 'M6', 'm7', 'M7']


---
## 4. SingleNoteGenerator

> **조회 방식**: `step_id` (전역 고유 ID) 하나만으로 O(1) 조회  
> `'CAT_SN_SC01_P01_S02'` → 단일음 > 7음계 > 파트01 > 스텝02 (악보)
>
> **생성 흐름**
> ```
> step_id → STEP_LOOKUP 조회  (O(1))
>        ↓
> answer_type      → ⑤ ANSWER_TYPE_RULES 참조
> difficulty_level → ⑥ DIFFICULTY_RULES  참조
>        ↓
> 유효성 검사 (pool_size vs num_choices)
>        ↓
> 정답음 선택 (pool + octave_range + 세션 이력 중복 체크)
>        ↓
> 오답 구성 (distractor_strategy + proximity_strategy)
>        ↓
> 종합난이도 계산 → Question dict 반환
> ```
>
> **오답 범위 규칙**  
> - note_pool 내 음이름에서만 선택  
> - 정답 기준 ±11반음(한 옥타브) 이내  
> - 오답 간 같은 음이름 중복 불가

In [12]:
class SingleNoteGenerator:
    """
    단일음 청음 문제 생성기

    Parameters
    ----------
    seed : int | None
        None → 매 실행마다 다른 결과 / 정수 → 재현 가능
    """

    def __init__(self, seed: Optional[int] = DEFAULT_SEED):
        self.rng = random.Random(seed)
        # 세션 이력: {step_id: [출제된 정답 MIDI 번호 리스트]}
        self._session_history: dict[str, list[int]] = {}

    # ── 퍼블릭 API ─────────────────────────────────────────────────────

    def generate(self, step_id: str, difficulty_level: int) -> dict:
        """
        문제 1개 생성

        Parameters
        ----------
        step_id          : 전역 고유 step ID
                           예) 'CAT_SN_SC01_P01_S02'
        difficulty_level : 난이도 레벨 1~3 (동적으로 전달)

        Returns
        -------
        dict:
          step_id, answer_type, difficulty_level,
          answer           : 정답 음이름+옥타브
          answer_midi      : 정답 MIDI 번호
          present_notes    : 제시음 리스트 (same_diff 는 2개)
          choices          : 선택지 리스트 (piano_subj 는 None)
          total_difficulty : 종합 난이도 (생성 후 분석 출력값)
        """
        step    = self._get_step(step_id)
        at_rule = self._get_answer_rule(step['answer_type'])
        d_rule  = self._get_difficulty_rule(difficulty_level)

        self._validate(step, at_rule)

        pool      = parse_pool(step['note_pool'])
        midi_pool = pool_to_midi_range(pool, d_rule['oct_min'], d_rule['oct_max'])
        answer_midi = self._pick_answer(step_id, midi_pool)
        answer      = midi_to_note(answer_midi)
        answer_type = step['answer_type']

        if answer_type == 'same_diff':
            present_notes, choices = self._build_same_diff(
                answer_midi, pool, d_rule['oct_min'], d_rule['oct_max']
            )
        elif answer_type == 'piano_subj':
            present_notes, choices = [answer], None
        else:  # score / note_name_Xchoice
            present_notes = [answer]
            choices = self._build_choices(
                answer_midi, pool,
                d_rule['oct_min'], d_rule['oct_max'],
                at_rule['num_choices'],
                at_rule['distractor_strategy'],
                d_rule['proximity_strategy'],
            )

        self._session_history.setdefault(step_id, []).append(answer_midi)

        return {
            'step_id':           step_id,
            'answer_type':       answer_type,
            'difficulty_level':  difficulty_level,
            'answer':            answer,
            'answer_midi':       answer_midi,
            'present_notes':     present_notes,
            'choices':           choices,
            'total_difficulty':  self._calc_difficulty(answer_type, difficulty_level),
        }

    def reset_session(self):
        """세션 이력 초기화 (새 세션 시작 시 호출)"""
        self._session_history = {}

    # ── 내부 로직 ───────────────────────────────────────────────────────

    def _get_step(self, step_id: str) -> dict:
        """전역 고유 step_id 로 커리큘럼 O(1) 조회"""
        if step_id not in STEP_LOOKUP:
            raise ValueError(
                f"step_id '{step_id}' 를 커리큘럼에서 찾을 수 없습니다.\n"
                f"형식 예시: 'CAT_SN_SC01_P01_S01'"
            )
        return STEP_LOOKUP[step_id]

    def _get_answer_rule(self, answer_type: str) -> dict:
        """⑤ answer_type 규칙 사전 조회"""
        if answer_type not in ANSWER_TYPE_RULES:
            raise ValueError(f"answer_type '{answer_type}' 이 규칙 사전에 없습니다.")
        keys = ['label','num_choices','present_count','distractor_strategy','pool_size_rule']
        return dict(zip(keys, ANSWER_TYPE_RULES[answer_type]))

    def _get_difficulty_rule(self, level: int) -> dict:
        """⑥ difficulty_level 규칙 사전 조회"""
        if level not in DIFFICULTY_RULES:
            raise ValueError(f"difficulty_level '{level}' 은 1~3 사이여야 합니다.")
        keys = ['label','oct_min','oct_max','proximity_strategy','proximity_semitones']
        return dict(zip(keys, DIFFICULTY_RULES[level]))

    def _validate(self, step: dict, at_rule: dict):
        """파라미터 조합 유효성 검사"""
        pool_size   = len(parse_pool(step['note_pool']))
        answer_type = step['answer_type']
        if answer_type == 'note_name_4choice' and pool_size < 4:
            raise ValueError(
                f"note_name_4choice 는 풀 크기 4 이상 필요. 현재: {pool_size}"
            )
        if answer_type == 'note_name_3choice' and pool_size != 3:
            raise ValueError(
                f"note_name_3choice 는 풀 크기 3 전용. 현재: {pool_size}"
            )

    def _pick_answer(self, step_id: str, midi_pool: list[int]) -> int:
        """세션 이력 체크 후 정답음 선택. 풀 소진 시 이력 리셋."""
        history    = self._session_history.get(step_id, [])
        candidates = [m for m in midi_pool if m not in history]
        if not candidates:
            print(f'  ⚠️  [{step_id}] 풀 소진 — 세션 이력 리셋')
            self._session_history[step_id] = []
            candidates = midi_pool
        return self.rng.choice(candidates)

    def _build_same_diff(
        self,
        answer_midi: int,
        pool: list[str],
        oct_min: int,
        oct_max: int,
    ) -> tuple[list, list]:
        """
        같음/다름 문제 생성
        첫 번째 음: 정답음 / 두 번째 음: 50% 확률로 같거나 pool 내 다른 음
        """
        first      = midi_to_note(answer_midi)
        is_same    = self.rng.choice([True, False])

        if is_same:
            second, label = first, '같음'
        else:
            answer_name = first[:-1]
            others = [
                m for m in pool_to_midi_range(pool, oct_min, oct_max)
                if midi_to_note(m)[:-1] != answer_name
            ]
            if not others:
                second, label = first, '같음'
            else:
                second, label = midi_to_note(self.rng.choice(others)), '다름'

        return [first, second], [label]

    def _build_choices(
        self,
        answer_midi: int,
        pool: list[str],
        oct_min: int,
        oct_max: int,
        num_choices: int,
        distractor_strategy: str,
        proximity_strategy: str,
    ) -> list[str]:
        """
        선다형 오답 구성
        - note_pool 내 음이름에서만 선택
        - 정답 기준 ±11반음(한 옥타브) 이내
        - 오답 간 같은 음이름 중복 불가
        """
        answer_name = midi_to_note(answer_midi)[:-1]
        all_midi    = pool_to_midi_range(pool, oct_min, oct_max)

        # 오답 후보: 정답 음이름 제외 + ±11반음 이내
        candidates = [
            m for m in all_midi
            if midi_to_note(m)[:-1] != answer_name
            and semitone_distance(m, answer_midi) <= 11
        ]

        # 풀이 작거나 use_all 이면 전체 사용, 아니면 proximity 적용
        if len(pool) <= 2 or distractor_strategy == 'use_all':
            distractor_pool = candidates
        else:
            distractor_pool = self._sort_by_proximity(
                answer_midi, candidates, proximity_strategy
            )

        distractors = self._pick_unique_name(
            distractor_pool, num_choices - 1
        )

        choices = [midi_to_note(answer_midi)] + [midi_to_note(d) for d in distractors]
        self.rng.shuffle(choices)
        return choices

    def _sort_by_proximity(
        self,
        answer_midi: int,
        candidates: list[int],
        strategy: str,
    ) -> list[int]:
        """
        proximity_strategy 에 따라 오답 후보 정렬
        asc_by_distance  : 가까운 음 우선 (어려움)
        desc_by_distance : 먼 음 우선 (쉬움)
        shuffle          : 무작위
        """
        if strategy == 'asc_by_distance':
            return sorted(candidates, key=lambda m: semitone_distance(m, answer_midi))
        elif strategy == 'desc_by_distance':
            return sorted(candidates, key=lambda m: semitone_distance(m, answer_midi), reverse=True)
        else:  # shuffle
            pool = candidates[:]
            self.rng.shuffle(pool)
            return pool

    def _pick_unique_name(self, candidates: list[int], n: int) -> list[int]:
        """
        오답 후보에서 음이름 중복 없이 n 개 선택
        (C3, C5 처럼 같은 음이름 다른 옥타브는 1개만 허용)
        """
        seen, result = set(), []
        for m in candidates:
            name = midi_to_note(m)[:-1]
            if name not in seen:
                seen.add(name)
                result.append(m)
            if len(result) == n:
                break
        if len(result) < n:
            print(f'  ⚠️  오답 후보 부족: {n}개 필요, {len(result)}개만 확보')
        return result

    def _calc_difficulty(self, answer_type: str, difficulty_level: int) -> float:
        """
        종합 난이도 계산 (생성 후 분석 출력값, 입력 파라미터 아님)
        선다형          : (oct_level + prox_level) / 2
        same_diff/subj  : oct_level 만 사용
        """
        if answer_type in ('same_diff', 'piano_subj'):
            return float(difficulty_level)
        return float(difficulty_level)  # oct == prox == difficulty_level 일 때

print('✅ SingleNoteGenerator 클래스 정의 완료')

✅ SingleNoteGenerator 클래스 정의 완료


---
## 5. IntervalGenerator


In [13]:
# ══════════════════════════════════════════════════════════════════════
# 섹션 8: IntervalGenerator — 음정 청음 문제 생성기
# ══════════════════════════════════════════════════════════════════════
class IntervalGenerator:
    """
    음정 청음 문제 생성기

    SingleNoteGenerator 와 공유:
      - 음 유틸: note_to_midi, midi_to_note
      - 음정 유틸: interval_semitones, build_interval_midi
      - 테이블: STEP_LOOKUP, ANSWER_TYPE_RULES, DIFFICULTY_RULES
      - 난이도 곡선: difficulty_curve()

    독립 구현:
      - 기준음 선택 (root_midi_pool 기반)
      - interval_pool 에서 음정 랜덤 출제
      - direction (ascending / descending / harmonic) 처리
      - 선다형: 음정이름 오답 구성 (반음 차이 기준)
      - height_compare: 같은 음정을 다른 높이 기준음으로 2회 제시
      - keyboard_subj: 음정이름+기준음 제시 → 건반 선택
    """

    def __init__(self, seed=None):
        self.rng = random.Random(seed)
        self._session_history: dict = {}

    # ── 퍼블릭 API ──────────────────────────────────────────────────

    def generate(self, step_id: str, difficulty_level: int) -> dict:
        """
        음정 문제 1개 생성

        Returns
        -------
        dict with keys:
          step_id, answer_type, difficulty_level, direction,
          answer_interval, answer_interval_ko,
          root_midi, root_note, upper_midi, upper_note,
          present_notes, choices, total_difficulty
        """
        step      = self._get_step(step_id)
        at_rule   = self._get_answer_rule(step['answer_type'])
        d_rule    = self._get_difficulty_rule(difficulty_level)
        direction = step['direction']
        ipool     = parse_interval_pool(step['note_pool'])

        root_midi, symbol = self._pick_root_and_interval(
            step_id, ipool, d_rule['oct_min'], d_rule['oct_max'], direction
        )
        root_note  = midi_to_note(root_midi)
        upper_midi = build_interval_midi(root_midi, symbol, direction)
        upper_note = midi_to_note(upper_midi)
        answer_type = step['answer_type']

        if answer_type == 'same_diff':
            present_notes, choices = self._build_same_diff(
                root_midi, symbol, ipool, d_rule, direction
            )
        elif answer_type == 'height_compare':
            present_notes, choices = self._build_height_compare(
                root_midi, symbol, d_rule, direction
            )
        elif answer_type in ('interval_subj', 'keyboard_subj'):
            if direction == 'descending':
                present_notes = [upper_note, root_note]
            else:
                present_notes = [root_note, upper_note]
            choices = None
        else:  # name_2/3/4choice
            if direction == 'descending':
                present_notes = [upper_note, root_note]
            else:
                present_notes = [root_note, upper_note]
            choices = self._build_name_choices(
                symbol, ipool, at_rule['num_choices'], d_rule['proximity_strategy']
            )

        self._session_history.setdefault(step_id, []).append((root_midi, symbol))

        return {
            'step_id':            step_id,
            'question_type':      'interval',
            'answer_type':        answer_type,
            'difficulty_level':   difficulty_level,
            'direction':          direction,
            'answer_interval':    symbol,
            'answer_interval_ko': interval_name_ko(symbol),
            'root_midi':          root_midi,
            'root_note':          root_note,
            'upper_midi':         upper_midi,
            'upper_note':         upper_note,
            'present_notes':      present_notes,
            'choices':            choices,
            'total_difficulty':   float(difficulty_level),
        }

    def reset_session(self):
        self._session_history = {}

    # ── 내부 로직 ───────────────────────────────────────────────────

    def _get_step(self, step_id):
        if step_id not in STEP_LOOKUP:
            raise ValueError(f"step_id '{step_id}' 를 찾을 수 없습니다.")
        return STEP_LOOKUP[step_id]

    def _get_answer_rule(self, answer_type):
        keys = ['label','num_choices','present_count','distractor_strategy','pool_size_rule']
        return dict(zip(keys, ANSWER_TYPE_RULES[answer_type]))

    def _get_difficulty_rule(self, level):
        keys = ['label','oct_min','oct_max','proximity_strategy','proximity_semitones']
        return dict(zip(keys, DIFFICULTY_RULES[level]))

    def _pick_root_and_interval(self, step_id, ipool, oct_min, oct_max, direction):
        # 세션 이력 체크 후 (기준음 MIDI, 음정 기호) 선택
        # 상단음이 MIDI 범위를 벗어나지 않도록 필터링
        history = self._session_history.get(step_id, [])
        candidates = [
            (r, sym)
            for r in root_midi_pool(oct_min, oct_max)
            for sym in ipool
            if MIDI_MIN <= build_interval_midi(r, sym, direction) <= MIDI_MAX
            and (r, sym) not in history
        ]
        if not candidates:
            print(f'  ⚠️  [{step_id}] 풀 소진 — 세션 이력 리셋')
            self._session_history[step_id] = []
            candidates = [
                (r, sym)
                for r in root_midi_pool(oct_min, oct_max)
                for sym in ipool
                if MIDI_MIN <= build_interval_midi(r, sym, direction) <= MIDI_MAX
            ]
        return self.rng.choice(candidates)

    def _build_same_diff(self, root_midi, symbol, ipool, d_rule, direction):
        # 같음/다름: 음정 2개 제시, 50% 확률로 같거나 다름
        is_same = self.rng.choice([True, False])
        p1_low, p1_high = interval_to_midi_pair(root_midi, symbol, direction)

        if is_same:
            label = '같음'
            p2_low, p2_high = p1_low, p1_high
        else:
            others = [s for s in ipool if s != symbol]
            if not others:
                label = '같음'
                p2_low, p2_high = p1_low, p1_high
            else:
                alt = self.rng.choice(others)
                alt_roots = [
                    r for r in root_midi_pool(d_rule['oct_min'], d_rule['oct_max'])
                    if MIDI_MIN <= build_interval_midi(r, alt, direction) <= MIDI_MAX
                ]
                alt_root = self.rng.choice(alt_roots) if alt_roots else root_midi
                p2_low, p2_high = interval_to_midi_pair(alt_root, alt, direction)
                label = '다름'

        present = [
            midi_to_note(p1_low), midi_to_note(p1_high),
            midi_to_note(p2_low), midi_to_note(p2_high),
        ]
        return present, [label]

    def _build_height_compare(self, root_midi, symbol, d_rule, direction):
        # 다양한 높이 비교: 같은 음정을 다른 높이에서 2회 제시
        alt_roots = [
            r for r in root_midi_pool(d_rule['oct_min'], d_rule['oct_max'])
            if MIDI_MIN <= build_interval_midi(r, symbol, direction) <= MIDI_MAX
            and r != root_midi
        ]
        alt_root = self.rng.choice(alt_roots) if alt_roots else root_midi
        p1_low, p1_high = interval_to_midi_pair(root_midi, symbol, direction)
        p2_low, p2_high = interval_to_midi_pair(alt_root,  symbol, direction)
        present = [
            midi_to_note(p1_low), midi_to_note(p1_high),
            midi_to_note(p2_low), midi_to_note(p2_high),
        ]
        return present, ['같음']

    def _build_name_choices(self, symbol, ipool, num_choices, proximity_strategy):
        # 음정이름 선다형 오답 구성
        # 오답: interval_pool 내에서 반음 차이 기준 정렬
        target_st = interval_semitones(symbol)
        pool_syms = [s for s in ipool if s != symbol]

        if proximity_strategy == 'asc_by_distance':
            pool_syms.sort(key=lambda s: abs(interval_semitones(s) - target_st))
        elif proximity_strategy == 'desc_by_distance':
            pool_syms.sort(key=lambda s: abs(interval_semitones(s) - target_st), reverse=True)
        else:
            self.rng.shuffle(pool_syms)

        distractors = pool_syms[:num_choices - 1]
        choices = [interval_name_ko(symbol)] + [interval_name_ko(s) for s in distractors]
        self.rng.shuffle(choices)

        if len(choices) < num_choices:
            print(f'  ⚠️  오답 후보 부족: {num_choices}개 필요, {len(choices)}개 확보')
        return choices

print('✅ IntervalGenerator 클래스 정의 완료')


✅ IntervalGenerator 클래스 정의 완료


---
## 6. 난이도 곡선 함수

> 세션 내 문제 순서별로 `difficulty_level` 배열을 생성합니다.
>
> | 모드 | 설명 | 10문제 예시 |
> |------|------|-------------|
> | `fixed`  | 전체 고정 — **일괄 난이도 지정 가능** | `[2,2,2,2,2,2,2,2,2,2]` |
> | `linear` | 선형 상승 | `[1,1,1,2,2,2,2,3,3,3]` |
> | `stairs` | 계단식 (기본 권장) | `[1,1,1,1,2,2,2,3,3,3]` |
> | `custom` | 사용자 정의 배열 | 직접 입력 |
>
> ⚠️ `fixed` 이외의 모드는 곡선 자동 배분이므로 `fixed_level` 과 동시에 사용 불가

In [14]:
def difficulty_curve(
    n_questions:   int,
    mode:          str = 'stairs',
    fixed_level:   int = 1,
    custom_levels: Optional[list[int]] = None,
) -> list[int]:
    """
    세션 내 문제 순서별 difficulty_level 배열 생성

    Parameters
    ----------
    n_questions   : 총 문제 수
    mode          : 'fixed' | 'linear' | 'stairs' | 'custom'
    fixed_level   : mode='fixed' 일 때 사용 (1~3)
    custom_levels : mode='custom' 일 때 직접 입력하는 배열
    """
    if mode == 'fixed':
        assert 1 <= fixed_level <= 3
        return [fixed_level] * n_questions

    elif mode == 'linear':
        import math
        return [min(1 + math.floor(i / n_questions * 3), 3)
                for i in range(n_questions)]

    elif mode == 'stairs':
        # 하→중→상  40% / 30% / 30%
        n1 = round(n_questions * 0.4)
        n3 = round(n_questions * 0.3)
        return [1]*n1 + [2]*(n_questions - n1 - n3) + [3]*n3

    elif mode == 'custom':
        assert custom_levels is not None
        assert len(custom_levels) == n_questions, \
            f'custom_levels 길이({len(custom_levels)}) ≠ n_questions({n_questions})'
        assert all(1 <= lv <= 3 for lv in custom_levels)
        return custom_levels

    else:
        raise ValueError(f"mode='{mode}' 은 'fixed'|'linear'|'stairs'|'custom' 중 하나")


# ── 곡선 미리보기 ──────────────────────────────────────────────────────
N = 10
df_curves = pd.DataFrame({
    'fixed(Lv.2)': difficulty_curve(N, mode='fixed',  fixed_level=2),
    'linear':      difficulty_curve(N, mode='linear'),
    'stairs':      difficulty_curve(N, mode='stairs'),
    'custom':      difficulty_curve(N, mode='custom',
                                    custom_levels=[1,1,2,1,3,2,3,2,3,3]),
}, index=[f'Q{i+1:02d}' for i in range(N)])

show(df_curves, '난이도 곡선 미리보기 (10문제 기준)', color='#1E8449')


── 난이도 곡선 미리보기 (10문제 기준) ──


,fixed(Lv.2),linear,stairs,custom
Q01,2,1,1,1
Q02,2,1,1,1
Q03,2,1,1,2
Q04,2,1,1,1
Q05,2,2,2,3
Q06,2,2,2,2
Q07,2,2,2,3
Q08,2,3,3,2
Q09,2,3,3,3
Q10,2,3,3,3


---
## 7. 통합 시뮬레이터

In [15]:
def simulate(
    step_id:       str,
    n_questions:   int  = 10,
    mode:          str  = 'stairs',
    fixed_level:   int  = 1,
    custom_levels: Optional[list[int]] = None,
    seed:          Optional[int] = DEFAULT_SEED,
    verbose:       bool = True,
) -> pd.DataFrame:
    """
    통합 세션 시뮬레이터 — 단일음(SingleNoteGenerator) + 음정(IntervalGenerator) 자동 선택

    Parameters
    ----------
    step_id      : 전역 고유 step ID
    n_questions  : 출제 문제 수
    mode         : 'fixed' | 'linear' | 'stairs' | 'custom'
    fixed_level  : mode='fixed' 일 때 난이도 (1~3)
    custom_levels: mode='custom' 일 때 직접 입력
    seed         : 재현 시드
    verbose      : True 면 세션 정보 + DataFrame 출력
    """
    step = STEP_LOOKUP[step_id]
    qtype = step['question_type']

    if qtype == 'single_note':
        gen = SingleNoteGenerator(seed=seed)
    elif qtype == 'interval':
        gen = IntervalGenerator(seed=seed)
    else:
        raise ValueError(f"알 수 없는 question_type: '{qtype}'")

    levels  = difficulty_curve(n_questions, mode=mode,
                               fixed_level=fixed_level, custom_levels=custom_levels)
    records = []
    for q_idx, level in enumerate(levels, 1):
        q = gen.generate(step_id=step_id, difficulty_level=level)
        diff_label = f"Lv.{q['difficulty_level']} ({DIFFICULTY_RULES[q['difficulty_level']][0]})"
        present_str = ' → '.join(q['present_notes']) if q.get('direction', '-') != 'harmonic' \
                      else ' + '.join(q['present_notes'])

        if qtype == 'interval':
            answer_str  = f"{q['answer_interval']} ({q['answer_interval_ko']})"
            choices_str = str(q['choices']) if q['choices'] else '(주관식)'
        else:
            answer_str  = q['answer']
            choices_str = str(q['choices']) if q['choices'] else '(주관식)'

        records.append({
            '#':           q_idx,
            'step_id':     q['step_id'],
            'answer_type': q['answer_type'],
            'direction':   q.get('direction', step.get('direction', '-')),
            'difficulty':  diff_label,
            'present':     present_str,
            'answer':      answer_str,
            'choices':     choices_str,
        })

    df_result = pd.DataFrame(records)

    if verbose:
        part_row   = next((p for p in PART_DATA   if p[1] == step['part_id']), None)
        course_row = next((c for c in COURSE_DATA if c[1] == (part_row[0] if part_row else '')), None)
        cat_row    = next((c for c in CATEGORY_DATA if c[0] == (course_row[0] if course_row else '')), None)
        print(f"\n{'═'*72}")
        print(f"  📋 세션 시뮬레이션 결과")
        print(f"  step_id   : {step_id}")
        print(f"  카테고리  : {cat_row[1] if cat_row else '-'}  |  코스: {course_row[2] if course_row else '-'}")
        print(f"  스텝명    : {step['step_name']}")
        pool_label = '음정풀' if qtype == 'interval' else '음풀'
        print(f"  {pool_label}     : {step['note_pool']}")
        print(f"  유형      : {step['answer_type']}  |  방향: {step.get('direction', '-')}  |  난이도 모드: {mode}")
        print(f"{'═'*72}")
        show(df_result, color='#2C3E50')

    return df_result

print('✅ 통합 시뮬레이터 정의 완료')


✅ 통합 시뮬레이터 정의 완료


### 6-1. 시뮬레이터 위젯 (ipywidgets / 변수 fallback)

In [16]:
# ── 선택 가능한 step_id 목록 (카테고리·코스·파트·스텝 계층) ─────────────
def _build_step_options():
    opts = {}
    for cat_id, cat_name in CATEGORY_DATA:
        for _, course_id, course_name in [c for c in COURSE_DATA if c[0]==cat_id]:
            for course_id2, part_id, part_name, *_ in [p for p in PART_DATA if p[0]==course_id]:
                for row in CURRICULUM_DATA:
                    if row[0] == part_id:
                        label = f"[{cat_name}] {course_name} > {part_name} > {row[2]} ({row[6]})"
                        opts[label] = row[1]  # step_id
    return opts

STEP_OPTIONS = _build_step_options()

if HAS_WIDGETS:
    # ── 위젯 UI ─────────────────────────────────────────────────────
    w_step     = widgets.Dropdown(
        options   = list(STEP_OPTIONS.items()),
        description='스텝 선택:',
        layout    = widgets.Layout(width='90%'),
        style     = {'description_width': '80px'},
    )
    w_n        = widgets.IntSlider(value=10, min=1, max=30, description='문제 수:',
                                   style={'description_width':'60px'})
    w_mode     = widgets.ToggleButtons(
        options     = ['fixed','linear','stairs','custom'],
        description = '난이도 모드:',
        style       = {'description_width':'80px', 'button_width':'80px'},
    )
    w_level    = widgets.IntSlider(value=1, min=1, max=3, description='고정 레벨:',
                                   style={'description_width':'70px'})
    w_seed     = widgets.IntText(value=0, description='Seed:',
                                 layout=widgets.Layout(width='180px'),
                                 style={'description_width':'50px'})
    w_run      = widgets.Button(description='▶ 시뮬레이션 실행',
                                button_style='success',
                                layout=widgets.Layout(width='200px'))
    out        = widgets.Output()

    def _on_mode_change(change):
        w_level.disabled = (change['new'] != 'fixed')
    w_mode.observe(_on_mode_change, names='value')

    def _on_run(_):
        with out:
            clear_output()
            step_id = w_step.value
            simulate(
                step_id       = step_id,
                n_questions   = w_n.value,
                mode          = w_mode.value,
                fixed_level   = w_level.value,
                seed          = w_seed.value,
                verbose       = True,
            )

    w_run.on_click(_on_run)

    display(widgets.VBox([
        widgets.HTML('<h4>🎵 청음 문제 시뮬레이터</h4>'),
        w_step,
        widgets.HBox([w_n, w_mode]),
        widgets.HBox([w_level, w_seed, w_run]),
        out,
    ]))

else:
    # ── fallback: 변수 직접 수정 ────────────────────────────────────
    print('ℹ️  ipywidgets 미설치 — 아래 변수를 직접 수정 후 셀을 실행하세요.')
    print()

    # ▼▼▼ 여기를 수정 ▼▼▼
    STEP_ID     = 'CAT_SN_SC01_P01_S02'   # step_id 직접 입력
    N_QUESTIONS = 10
    MODE        = 'stairs'                 # 'fixed' | 'linear' | 'stairs' | 'custom'
    FIXED_LEVEL = 1
    SEED        = 0
    # ▲▲▲ 여기까지 ▲▲▲

    simulate(
        step_id     = STEP_ID,
        n_questions = N_QUESTIONS,
        mode        = MODE,
        fixed_level = FIXED_LEVEL,
        seed        = SEED,
        verbose     = True,
    )


### 6-2. 단일음 예시

In [22]:
# 단일음 예시: SC01 P03 음이름(4지) — 계단식
simulate('CAT_SN_SC01_P03_S04', n_questions=8, mode='stairs', seed=42)
print('✅done')


════════════════════════════════════════════════════════════════════════
  📋 세션 시뮬레이션 결과
  step_id   : CAT_SN_SC01_P03_S04
  카테고리  : 단일음  |  코스: 7음계
  스텝명    : 음이름(4지)
  음풀     : C,D,F,G
  유형      : name_4choice  |  방향: -  |  난이도 모드: stairs
════════════════════════════════════════════════════════════════════════


,#,step_id,answer_type,direction,difficulty,present,answer,choices
0,1,CAT_SN_SC01_P03_S04,name_4choice,-,Lv.1 (쉬움),C4,C4,"['D4', 'G4', 'F4', 'C4']"
1,2,CAT_SN_SC01_P03_S04,name_4choice,-,Lv.1 (쉬움),D4,D4,"['C4', 'F4', 'D4', 'G4']"
2,3,CAT_SN_SC01_P03_S04,name_4choice,-,Lv.1 (쉬움),F4,F4,"['C4', 'D4', 'F4', 'G4']"
3,4,CAT_SN_SC01_P03_S04,name_4choice,-,Lv.2 (보통),D3,D3,"['D3', 'F3', 'G3', 'C4']"
4,5,CAT_SN_SC01_P03_S04,name_4choice,-,Lv.2 (보통),G4,G4,"['G4', 'F5', 'D4', 'C4']"
5,6,CAT_SN_SC01_P03_S04,name_4choice,-,Lv.2 (보통),F3,F3,"['C4', 'D4', 'F3', 'G3']"
6,7,CAT_SN_SC01_P03_S04,name_4choice,-,Lv.3 (어려움),C6,C6,"['C6', 'G5', 'D5', 'F5']"
7,8,CAT_SN_SC01_P03_S04,name_4choice,-,Lv.3 (어려움),C3,C3,"['D3', 'C3', 'F3', 'G3']"


✅done


In [21]:
# 단일음 예시: SC02 P06 음이름(3지/A,Bb,B) — 선형
simulate('CAT_SN_SC02_P06_S03', n_questions=6, mode='linear', seed=7)
print('✅done')


════════════════════════════════════════════════════════════════════════
  📋 세션 시뮬레이션 결과
  step_id   : CAT_SN_SC02_P06_S03
  카테고리  : 단일음  |  코스: 12음계
  스텝명    : 음이름(3지)
  음풀     : A,Bb,B
  유형      : name_3choice  |  방향: -  |  난이도 모드: linear
════════════════════════════════════════════════════════════════════════


,#,step_id,answer_type,direction,difficulty,present,answer,choices
0,1,CAT_SN_SC02_P06_S03,name_3choice,-,Lv.1 (쉬움),Bb4,Bb4,"['B4', 'A4', 'Bb4']"
1,2,CAT_SN_SC02_P06_S03,name_3choice,-,Lv.1 (쉬움),A4,A4,"['Bb4', 'B4', 'A4']"
2,3,CAT_SN_SC02_P06_S03,name_3choice,-,Lv.2 (보통),B3,B3,"['A3', 'B3', 'Bb3']"
3,4,CAT_SN_SC02_P06_S03,name_3choice,-,Lv.2 (보통),Bb5,Bb5,"['B4', 'A5', 'Bb5']"
4,5,CAT_SN_SC02_P06_S03,name_3choice,-,Lv.3 (어려움),A3,A3,"['A3', 'B3', 'Bb3']"
5,6,CAT_SN_SC02_P06_S03,name_3choice,-,Lv.3 (어려움),Bb3,Bb3,"['A3', 'B3', 'Bb3']"


✅done


### 6-3. 음정 예시

In [23]:
# 음정 예시: SC01 P01 같음/다름 (완전1도, 장2도)
simulate('CAT_INT_SC01_P01_S01', n_questions=6, mode='fixed', fixed_level=1, seed=1)
print('✅done')


════════════════════════════════════════════════════════════════════════
  📋 세션 시뮬레이션 결과
  step_id   : CAT_INT_SC01_P01_S01
  카테고리  : 음정  |  코스: 코스1: 1도, 2도
  스텝명    : 음정 같음/다름
  음정풀     : P1,M2
  유형      : same_diff  |  방향: ascending  |  난이도 모드: fixed
════════════════════════════════════════════════════════════════════════


,#,step_id,answer_type,direction,difficulty,present,answer,choices
0,1,CAT_INT_SC01_P01_S01,same_diff,ascending,Lv.1 (쉬움),D4 → D4 → D4 → D4,P1 (완전1도),['같음']
1,2,CAT_INT_SC01_P01_S01,same_diff,ascending,Lv.1 (쉬움),E4 → F#4 → E4 → F#4,M2 (장2도),['같음']
2,3,CAT_INT_SC01_P01_S01,same_diff,ascending,Lv.1 (쉬움),G#4 → Bb4 → Bb4 → Bb4,M2 (장2도),['다름']
3,4,CAT_INT_SC01_P01_S01,same_diff,ascending,Lv.1 (쉬움),G4 → G4 → G4 → G4,P1 (완전1도),['같음']
4,5,CAT_INT_SC01_P01_S01,same_diff,ascending,Lv.1 (쉬움),C#4 → Eb4 → F#4 → F#4,M2 (장2도),['다름']
5,6,CAT_INT_SC01_P01_S01,same_diff,ascending,Lv.1 (쉬움),A4 → A4 → A4 → A4,P1 (완전1도),['같음']


✅done


In [26]:
# 음정 예시: SC02 P03 음정 이름 고르기 — 상행 (단3도, 장3도)
simulate('CAT_INT_SC02_P03_S03', n_questions=8, mode='stairs', seed=10)
print('✅done')


════════════════════════════════════════════════════════════════════════
  📋 세션 시뮬레이션 결과
  step_id   : CAT_INT_SC02_P03_S03
  카테고리  : 음정  |  코스: 코스2: 2도, 3도
  스텝명    : 음정 이름 고르기
  음정풀     : m3,M3
  유형      : name_2choice  |  방향: ascending  |  난이도 모드: stairs
════════════════════════════════════════════════════════════════════════


,#,step_id,answer_type,direction,difficulty,present,answer,choices
0,1,CAT_INT_SC02_P03_S03,name_2choice,ascending,Lv.1 (쉬움),A4 → C5,m3 (단3도),"['장3도', '단3도']"
1,2,CAT_INT_SC02_P03_S03,name_2choice,ascending,Lv.1 (쉬움),F#4 → Bb4,M3 (장3도),"['장3도', '단3도']"
2,3,CAT_INT_SC02_P03_S03,name_2choice,ascending,Lv.1 (쉬움),Bb4 → C#5,m3 (단3도),"['장3도', '단3도']"
3,4,CAT_INT_SC02_P03_S03,name_2choice,ascending,Lv.2 (보통),C#4 → E4,m3 (단3도),"['단3도', '장3도']"
4,5,CAT_INT_SC02_P03_S03,name_2choice,ascending,Lv.2 (보통),E5 → G5,m3 (단3도),"['단3도', '장3도']"
5,6,CAT_INT_SC02_P03_S03,name_2choice,ascending,Lv.2 (보통),E5 → G#5,M3 (장3도),"['장3도', '단3도']"
6,7,CAT_INT_SC02_P03_S03,name_2choice,ascending,Lv.3 (어려움),Bb4 → D5,M3 (장3도),"['단3도', '장3도']"
7,8,CAT_INT_SC02_P03_S03,name_2choice,ascending,Lv.3 (어려움),C#3 → E3,m3 (단3도),"['단3도', '장3도']"


✅done


In [25]:
# 음정 예시: SC03 P04 전체복습 — 하행 주관식
simulate('CAT_INT_SC03_P04_S03', n_questions=6, mode='fixed', fixed_level=2, seed=5)
print('✅done')


════════════════════════════════════════════════════════════════════════
  📋 세션 시뮬레이션 결과
  step_id   : CAT_INT_SC03_P04_S03
  카테고리  : 음정  |  코스: 코스3: 3도, 4도
  스텝명    : 하행 음정 알아맞히기
  음정풀     : m3,M3,P4,A4
  유형      : interval_subj  |  방향: descending  |  난이도 모드: fixed
════════════════════════════════════════════════════════════════════════


,#,step_id,answer_type,direction,difficulty,present,answer,choices
0,1,CAT_INT_SC03_P04_S03,interval_subj,descending,Lv.2 (보통),G#4 → C5,M3 (장3도),(주관식)
1,2,CAT_INT_SC03_P04_S03,interval_subj,descending,Lv.2 (보통),G3 → C4,P4 (완전4도),(주관식)
2,3,CAT_INT_SC03_P04_S03,interval_subj,descending,Lv.2 (보통),B4 → E5,P4 (완전4도),(주관식)
3,4,CAT_INT_SC03_P04_S03,interval_subj,descending,Lv.2 (보통),C#4 → E4,m3 (단3도),(주관식)
4,5,CAT_INT_SC03_P04_S03,interval_subj,descending,Lv.2 (보통),C5 → F#5,A4 (증4도),(주관식)
5,6,CAT_INT_SC03_P04_S03,interval_subj,descending,Lv.2 (보통),B4 → Eb5,M3 (장3도),(주관식)


✅done


In [24]:
# 음정 예시: SC05 P04 전체복습 — 화음 주관식
simulate('CAT_INT_SC05_P04_S05', n_questions=6, mode='fixed', fixed_level=2, seed=3)
print('✅done')


════════════════════════════════════════════════════════════════════════
  📋 세션 시뮬레이션 결과
  step_id   : CAT_INT_SC05_P04_S05
  카테고리  : 음정  |  코스: 코스5: 3도, 6도
  스텝명    : 화음에서 음정 찾기
  음정풀     : m3,M3,m6,M6
  유형      : interval_subj  |  방향: harmonic  |  난이도 모드: fixed
════════════════════════════════════════════════════════════════════════


,#,step_id,answer_type,direction,difficulty,present,answer,choices
0,1,CAT_INT_SC05_P04_S05,interval_subj,harmonic,Lv.2 (보통),G3 + Eb4,m6 (단6도),(주관식)
1,2,CAT_INT_SC05_P04_S05,interval_subj,harmonic,Lv.2 (보통),G4 + Bb4,m3 (단3도),(주관식)
2,3,CAT_INT_SC05_P04_S05,interval_subj,harmonic,Lv.2 (보통),F4 + C#5,m6 (단6도),(주관식)
3,4,CAT_INT_SC05_P04_S05,interval_subj,harmonic,Lv.2 (보통),E3 + G3,m3 (단3도),(주관식)
4,5,CAT_INT_SC05_P04_S05,interval_subj,harmonic,Lv.2 (보통),C4 + E4,M3 (장3도),(주관식)
5,6,CAT_INT_SC05_P04_S05,interval_subj,harmonic,Lv.2 (보통),G#5 + C6,M3 (장3도),(주관식)


✅done


---
## 8. Exhaustive 검증기 (테스트/버그 확인용)

> **목표**: 모든 `step_id × difficulty_level(1~3) × 정답음(MIDI 풀 전체)` 조합을 강제 생성하고,
> 각 문제마다 아래 5가지 규칙을 자동 검증합니다.
>
> | 검증 항목 | 규칙 | 적용 대상 |
> |---------|------|----------|
> | ① pool_answer  | 정답 음이름이 note_pool 내에 있어야 함 | 전체 |
> | ② pool_choices | 모든 선택지 음이름이 note_pool 내에만 있어야 함 | 선다형만 |
> | ③ no_dup_name  | 오답 간 같은 음이름 중복 없어야 함 | 선다형만 |
> | ④ choice_count | 선택지 수가 num_choices와 일치해야 함 | 선다형만 |
> | ⑤ midi_range   | 모든 제시음·선택지 MIDI가 48~84 이내여야 함 | 전체 |
>
> **강제 생성 방식**: `_session_history` 에 target_midi 외 전부 선입력
> → `_pick_answer` 가 반드시 target_midi 를 정답으로 선택  
> **예상 조합 수**: step(68) × difficulty(3) × 스텝별 MIDI 풀 크기 ≈ **2,383개**
>
> ⚠️ `same_diff` 는 choices 가 `['같음'/'다름']` 라벨이므로 ②③④ 검증 스킵

In [28]:
# ══════════════════════════════════════════════════════════════════════
# 섹션 9: Exhaustive 검증기 — 단일음 + 음정 통합
# ══════════════════════════════════════════════════════════════════════

RULE_KEYS = ['① pool_answer','② pool_choices','③ no_dup_name','④ choice_count','⑤ midi_range']

def validate_question(q: dict, pool_str: str, num_choices: int) -> dict:
    """
    단일음 / 음정 문제 공통 검증기

    검증 규칙
    ---------
    ① pool_answer  : 정답(음이름 or 음정기호)이 pool 내에 있어야 함
    ② pool_choices : 선택지가 pool 내 항목만 포함해야 함 (선다형만)
    ③ no_dup_name  : 오답 간 중복 없어야 함 (선다형만)
    ④ choice_count : 선택지 수가 num_choices 와 일치해야 함 (선다형만)
    ⑤ midi_range   : 모든 present_notes MIDI 가 MIDI_MIN~MIDI_MAX 이내여야 함
    """
    qtype       = q['question_type'] if 'question_type' in q else STEP_LOOKUP[q['step_id']]['question_type']
    answer_type = q['answer_type']
    errors      = []
    result      = {}

    if qtype == 'single_note':
        pool_set    = set(parse_pool(pool_str))
        answer_name = q['answer'][:-1]
        # ①
        ok1 = answer_name in pool_set
        result['① pool_answer'] = ok1
        if not ok1: errors.append(f"정답 음이름 '{answer_name}' 이 pool {list(pool_set)} 에 없음")
        choices = q.get('choices')
        if answer_type in ('same_diff','piano_subj') or choices is None:
            result['② pool_choices'] = True; result['③ no_dup_name'] = True; result['④ choice_count'] = True
            oor = [n for n in q['present_notes'] if not (MIDI_MIN <= note_to_midi(n) <= MIDI_MAX)]
            result['⑤ midi_range'] = len(oor)==0
            if oor: errors.append(f"MIDI 범위 초과: {oor}")
        else:
            bad = [c for c in choices if c[:-1] not in pool_set]
            result['② pool_choices'] = len(bad)==0
            if bad: errors.append(f"pool 외 선택지: {bad}")
            dnames = [c[:-1] for c in choices if c[:-1] != answer_name]
            ok3 = len(dnames)==len(set(dnames))
            result['③ no_dup_name'] = ok3
            if not ok3: errors.append(f"오답 중복: {list({n for n in dnames if dnames.count(n)>1})}")
            result['④ choice_count'] = len(choices)==num_choices
            if len(choices)!=num_choices: errors.append(f"선택지 수 불일치: {len(choices)}≠{num_choices}")
            oor = [n for n in q['present_notes'] + choices if not (MIDI_MIN <= note_to_midi(n) <= MIDI_MAX)]
            result['⑤ midi_range'] = len(oor)==0
            if oor: errors.append(f"MIDI 범위 초과: {oor}")

    else:  # interval
        ipool_syms  = parse_interval_pool(pool_str)
        ipool_set   = set(ipool_syms)
        answer_sym  = q['answer_interval']
        # ①
        ok1 = answer_sym in ipool_set
        result['① pool_answer'] = ok1
        if not ok1: errors.append(f"정답 음정 '{answer_sym}' 이 pool {ipool_syms} 에 없음")
        choices = q.get('choices')
        if answer_type in ('same_diff','height_compare','interval_subj','keyboard_subj') or choices is None:
            result['② pool_choices'] = True; result['③ no_dup_name'] = True; result['④ choice_count'] = True
        else:
            # choices 는 한국어 이름 리스트
            pool_ko  = {interval_name_ko(s) for s in ipool_syms}
            bad  = [c for c in choices if c not in pool_ko]
            result['② pool_choices'] = len(bad)==0
            if bad: errors.append(f"pool 외 선택지: {bad}")
            ans_ko  = interval_name_ko(answer_sym)
            dnames  = [c for c in choices if c != ans_ko]
            ok3 = len(dnames)==len(set(dnames))
            result['③ no_dup_name'] = ok3
            if not ok3: errors.append(f"오답 중복: {list({n for n in dnames if dnames.count(n)>1})}")
            result['④ choice_count'] = len(choices)==num_choices
            if len(choices)!=num_choices: errors.append(f"선택지 수 불일치: {len(choices)}≠{num_choices}")
        # ⑤ present_notes MIDI 범위
        try:
            oor = [n for n in q['present_notes'] if not (MIDI_MIN <= note_to_midi(n) <= MIDI_MAX)]
            result['⑤ midi_range'] = len(oor)==0
            if oor: errors.append(f"MIDI 범위 초과: {oor}")
        except Exception as e:
            result['⑤ midi_range'] = False; errors.append(f"MIDI 검증 오류: {e}")

    result['errors'] = errors
    result['passed'] = all(result[k] for k in RULE_KEYS)
    return result

print('✅ validate_question() 정의 완료 (단일음 + 음정 통합)')


✅ validate_question() 정의 완료 (단일음 + 음정 통합)


In [29]:
def run_exhaustive_validation(
    seed: Optional[int] = 0,
    verbose_fail: bool  = True,
    category: str       = 'all',   # 'all' | 'single_note' | 'interval'
) -> pd.DataFrame:
    """
    모든 step × difficulty × 정답(pool 전체) 조합 생성 + 검증

    Parameters
    ----------
    seed         : 재현 시드
    verbose_fail : True 면 실패 케이스 즉시 출력
    category     : 'all' | 'single_note' | 'interval'  — 대상 필터
    """
    records              = []
    total = pass_cnt = fail_cnt = 0

    for step_id, step in STEP_LOOKUP.items():
        qtype = step['question_type']
        if category != 'all' and qtype != category:
            continue

        pool_str    = step['note_pool']
        answer_type = step['answer_type']
        num_choices = dict(zip(
            ['label','num_choices','present_count','distractor_strategy','pool_size_rule'],
            ANSWER_TYPE_RULES[answer_type]
        ))['num_choices']
        direction = step['direction']

        for diff_level in [1, 2, 3]:
            _, oct_min, oct_max, *_ = DIFFICULTY_RULES[diff_level]

            if qtype == 'single_note':
                pool      = parse_pool(pool_str)
                midi_pool = pool_to_midi_range(pool, oct_min, oct_max)
                targets   = [(m, None) for m in midi_pool]
            else:  # interval
                ipool  = parse_interval_pool(pool_str)
                targets = [
                    (r, sym)
                    for r in root_midi_pool(oct_min, oct_max)
                    for sym in ipool
                    if MIDI_MIN <= build_interval_midi(r, sym, direction) <= MIDI_MAX
                ]

            for target in targets:
                if qtype == 'single_note':
                    target_midi, _ = target
                    gen = SingleNoteGenerator(seed=seed)
                    gen._session_history[step_id] = [m for m in midi_pool if m != target_midi]
                    target_label = midi_to_note(target_midi)
                else:
                    target_root, target_sym = target
                    gen = IntervalGenerator(seed=seed)
                    gen._session_history[step_id] = [
                        t for t in [
                            (r, sym)
                            for r in root_midi_pool(oct_min, oct_max)
                            for sym in ipool
                            if MIDI_MIN <= build_interval_midi(r, sym, direction) <= MIDI_MAX
                        ]
                        if t != target
                    ]
                    target_label = f"{midi_to_note(target_root)}+{target_sym}"

                try:
                    q      = gen.generate(step_id=step_id, difficulty_level=diff_level)
                    # question_type 주입 (validate_question 에서 사용)
                    q['question_type'] = qtype
                    v      = validate_question(q, pool_str, num_choices)
                    passed = v['passed']
                    errors = ' | '.join(v['errors'])
                except Exception as e:
                    passed = False
                    errors = f'[EXCEPTION] {e}'
                    v      = {k: False for k in RULE_KEYS}

                total += 1
                if passed: pass_cnt += 1
                else:
                    fail_cnt += 1
                    if verbose_fail:
                        print(f'  ❌ {step_id} | diff={diff_level} | target={target_label} | {errors}')

                records.append({
                    'step_id':        step_id,
                    'question_type':  qtype,
                    'answer_type':    answer_type,
                    'direction':      direction,
                    'note_pool':      pool_str,
                    'diff_level':     diff_level,
                    'target':         target_label,
                    'passed':         passed,
                    '① pool_answer':  v.get('① pool_answer',  '-'),
                    '② pool_choices': v.get('② pool_choices', '-'),
                    '③ no_dup_name':  v.get('③ no_dup_name',  '-'),
                    '④ choice_count': v.get('④ choice_count', '-'),
                    '⑤ midi_range':   v.get('⑤ midi_range',   '-'),
                    'errors':         errors,
                })

    df_all = pd.DataFrame(records)
    print(f"\n{'═'*65}")
    print(f"  🧪 Exhaustive 검증 완료  (대상: {category})")
    print(f"  총 조합 수  : {total:,} 개")
    print(f"  ✅ 통과     : {pass_cnt:,} 개  ({pass_cnt/max(total,1)*100:.1f}%)")
    print(f"  ❌ 실패     : {fail_cnt:,} 개  ({fail_cnt/max(total,1)*100:.1f}%)")
    print(f"{'═'*65}")
    for col in RULE_KEYS:
        n_fail = (df_all[col] == False).sum()
        if n_fail > 0: print(f"  {col} 실패: {n_fail}건")
    if fail_cnt == 0: print("  🎉 모든 조합 통과!")
    return df_all

print('✅ run_exhaustive_validation() 정의 완료')


✅ run_exhaustive_validation() 정의 완료


In [34]:
# ── 검증 실행 ──────────────────────────────────────────────────────────
# verbose_fail=True  → 실패 케이스 실시간 출력
# verbose_fail=False → 조용히 돌리고 요약만 출력

df_validation = run_exhaustive_validation(seed=0, verbose_fail=True, category='single_note')


═════════════════════════════════════════════════════════════════
  🧪 Exhaustive 검증 완료  (대상: single_note)
  총 조합 수  : 2,383 개
  ✅ 통과     : 2,383 개  (100.0%)
  ❌ 실패     : 0 개  (0.0%)
═════════════════════════════════════════════════════════════════
  🎉 모든 조합 통과!


In [35]:
# ── 실패 케이스만 추출 ─────────────────────────────────────────────────

df_failed = df_validation[df_validation['passed'] == False].reset_index(drop=True)

if len(df_failed) == 0:
    print('🎉 실패 케이스 없음 — 모든 조합 통과!')
else:
    print(f'❌ 실패 케이스 {len(df_failed)}건')
    show(df_failed, f'실패 케이스 ({len(df_failed)}건)', color='#C0392B')

🎉 실패 케이스 없음 — 모든 조합 통과!


In [36]:
# ── 전체 결과 통계 ─────────────────────────────────────────────────────
# step_id × answer_type × diff_level 별 통과율 집계

df_summary = (
    df_validation
    .groupby(['step_id', 'answer_type', 'diff_level'])
    .agg(
        total  = ('passed', 'count'),
        passed = ('passed', 'sum'),
        failed = ('passed', lambda x: (~x).sum()),
    )
    .reset_index()
)
df_summary['pass_rate'] = (
    df_summary['passed'] / df_summary['total'] * 100
).round(1).astype(str) + '%'

df_has_fail = df_summary[df_summary['failed'] > 0]
if len(df_has_fail) > 0:
    show(df_has_fail, '실패 있는 스텝 요약', color='#C0392B')
else:
    print('🎉 모든 step × diff 조합 100% 통과!')
    show(df_summary, '전체 검증 요약 (step × diff_level)', color='#1E8449')

🎉 모든 step × diff 조합 100% 통과!

── 전체 검증 요약 (step × diff_level) ──


,step_id,answer_type,diff_level,total,passed,failed,pass_rate
0,CAT_SN_SC01_P01_S01,same_diff,1,2,2,0,100.0%
1,CAT_SN_SC01_P01_S01,same_diff,2,6,6,0,100.0%
2,CAT_SN_SC01_P01_S01,same_diff,3,7,7,0,100.0%
3,CAT_SN_SC01_P01_S02,name_2choice,1,2,2,0,100.0%
4,CAT_SN_SC01_P01_S02,name_2choice,2,6,6,0,100.0%
5,CAT_SN_SC01_P01_S02,name_2choice,3,7,7,0,100.0%
6,CAT_SN_SC01_P01_S03,name_2choice,1,2,2,0,100.0%
7,CAT_SN_SC01_P01_S03,name_2choice,2,6,6,0,100.0%
8,CAT_SN_SC01_P01_S03,name_2choice,3,7,7,0,100.0%
9,CAT_SN_SC01_P01_S04,piano_subj,1,2,2,0,100.0%
